## 2点相関関数（Landy-Szalay推定量）

Landy-Szalay (1993) 推定量は以下で定義される：

$$\hat{\xi}(r) = \frac{DD(r) - 2\,DR(r) + RR(r)}{RR(r)}$$

各項は正規化されたペアカウント：

$$DD(r) = \frac{n_{DD}(r)}{N_D(N_D-1)/2}, \quad DR(r) = \frac{n_{DR}(r)}{N_D N_R}, \quad RR(r) = \frac{n_{RR}(r)}{N_R(N_R-1)/2}$$

- $N_D$：データ点数（Colesの店舗数）
- $N_R$：ランダムカタログのサイズ（$N_R \gg N_D$）
- $r$：ハーバーサイン距離 [km]

ランダムカタログはオーストラリアのバウンディングボックス内で一様サンプリング。
Landy-Szalay推定量の分散の下限（Poisson）は $\text{Var}[\hat{\xi}] \approx (1 + \hat{\xi})^2 / n_{RR}$。

In [1]:
%use dataframe
%use lets-plot

In [2]:
import kotlin.math.*
import kotlin.random.Random

// --- データ読み込み ---
val df = DataFrame.readCSV("./output/coles_locations.csv")

data class Point(val lat: Double, val lon: Double)

val dataPoints: List<Point> = df.rows().map { row ->
    Point(lat = row["lat"] as Double, lon = row["lon"] as Double)
}
val nD = dataPoints.size
println("N_D = $nD")

N_D = 685


In [3]:
// --- ハーバーサイン距離 [km] ---
fun haversine(p1: Point, p2: Point): Double {
    val R = 6371.0
    val dLat = Math.toRadians(p2.lat - p1.lat)
    val dLon = Math.toRadians(p2.lon - p1.lon)
    val sinLat = sin(dLat / 2)
    val sinLon = sin(dLon / 2)
    val a = sinLat * sinLat +
            cos(Math.toRadians(p1.lat)) * cos(Math.toRadians(p2.lat)) * sinLon * sinLon
    return 2.0 * R * asin(sqrt(a))
}

In [4]:
// --- 平均最近傍距離の計算（ビン設計の基準スケール）---
// O(N^2) だが N=685 なので数秒で完了
val nnDistances = dataPoints.map { p1 ->
    dataPoints.filter { it !== p1 }.minOf { p2 -> haversine(p1, p2) }
}
val meanNN = nnDistances.average()
val minNN  = nnDistances.min()
val maxNN  = nnDistances.max()

println("最近傍距離 [km]:")
println("  平均 (mean) = %.2f".format(meanNN))
println("  最小 (min)  = %.2f".format(minNN))
println("  最大 (max)  = %.2f".format(maxNN))

最近傍距離 [km]:
  平均 (mean) = 15.25
  最小 (min)  = 0.17
  最大 (max)  = 665.86


In [5]:
// --- 距離ビンの設計 ---
// rMin: 平均最近傍距離の 1/2（クラスタリングスケールの下限を確実に捉える）
// rMax: 大陸スケール上限 [km]
// nBins: Δln(r) ≈ 0.15 を目標に逆算
val rMin  = meanNN / 2.0
val rMax  = 2000.0
val dLogR = 0.15  // 対数ビン幅の目標値
val nBins = kotlin.math.ceil(kotlin.math.ln(rMax / rMin) / dLogR).toInt()

val bins = DoubleArray(nBins + 1) { i ->
    rMin * (rMax / rMin).pow(i.toDouble() / nBins)
}
val rCenters = DoubleArray(nBins) { i -> kotlin.math.sqrt(bins[i] * bins[i + 1]) }

println("rMin  = %.2f km  (= meanNN / 2)".format(rMin))
println("rMax  = %.1f km".format(rMax))
println("nBins = %d  (Δln r = %.3f)".format(nBins, kotlin.math.ln(rMax / rMin) / nBins))
println("ビン端点: ${bins.take(4).map { "%.1f".format(it) }} ... ${"%.1f".format(bins.last())} km")

rMin  = 7.63 km  (= meanNN / 2)
rMax  = 2000.0 km
nBins = 38  (Δln r = 0.147)
ビン端点: [7.6, 8.8, 10.2, 11.8] ... 2000.0 km


In [6]:
// --- ランダムカタログ生成（オーストラリア・バウンディングボックス） ---
// 注: 大陸マスクなしの矩形一様サンプリング。海洋点を含むが
//     Landy-Szalay推定量はこの境界効果を RR で相殺する。
val rng = Random(42)
val latMin = -44.0; val latMax = -10.0
val lonMin = 113.0; val lonMax = 154.0
val nR = nD * 10   // N_R >> N_D で RR のショットノイズを抑制

val randomPoints = List(nR) {
    Point(
        lat = latMin + rng.nextDouble() * (latMax - latMin),
        lon = lonMin + rng.nextDouble() * (lonMax - lonMin)
    )
}
println("N_R = $nR")

N_R = 6850


In [7]:
// --- ペアカウント関数 ---
// points2 == null のとき自己ペア（i < j のみ）をカウント
fun pairCounts(
    points1: List<Point>,
    points2: List<Point>?,
    bins: DoubleArray
): LongArray {
    val counts = LongArray(bins.size - 1)
    val pts2 = points2 ?: points1
    val selfPairs = points2 == null

    for (i in points1.indices) {
        val jStart = if (selfPairs) i + 1 else 0
        for (j in jStart until pts2.size) {
            val d = haversine(points1[i], pts2[j])
            // 二分探索でビンを特定
            var lo = 0; var hi = bins.size - 1
            while (lo < hi - 1) {
                val mid = (lo + hi) / 2
                if (bins[mid] <= d) lo = mid else hi = mid
            }
            if (d >= bins[lo] && d < bins[hi]) counts[lo]++
        }
    }
    return counts
}

// --- DD, DR, RR の計算（※ RR は N_R^2/2 ≈ 2.3×10^7 ペアで数分かかる） ---
println("DD を計算中...")
val nDD = pairCounts(dataPoints, null, bins)

println("DR を計算中...")
val nDR = pairCounts(dataPoints, randomPoints, bins)

println("RR を計算中...")
val nRR = pairCounts(randomPoints, null, bins)

println("完了")

DD を計算中...
DR を計算中...
RR を計算中...
完了


In [8]:
// --- Landy-Szalay 推定量 ---
val normDD = nD.toLong() * (nD - 1) / 2
val normDR = nD.toLong() * nR
val normRR = nR.toLong() * (nR - 1) / 2

data class XiBin(val rCenter: Double, val xi: Double, val xiErr: Double, val nRR: Long)

val xiResult: List<XiBin> = (0 until nBins).map { i ->
    val dd = nDD[i].toDouble() / normDD
    val dr = nDR[i].toDouble() / normDR
    val rr = nRR[i].toDouble() / normRR

    val xi = if (rr > 0.0) (dd - 2.0 * dr + rr) / rr else Double.NaN
    // Poisson誤差の下限: σ_ξ ≈ (1 + ξ) / sqrt(n_RR)
    val xiErr = if (nRR[i] > 0) (1.0 + xi) / sqrt(nRR[i].toDouble()) else Double.NaN

    XiBin(rCenters[i], xi, xiErr, nRR[i])
}

xiResult.forEach { b ->
    println("r = %7.1f km  ξ = %+.4f ± %.4f  (n_RR = %d)".format(b.rCenter, b.xi, b.xiErr, b.nRR))
}

r =     8.2 km  ξ = +519.0469 ± 54.5157  (n_RR = 91)
r =     9.5 km  ξ = +553.9043 ± 51.7451  (n_RR = 115)
r =    11.0 km  ξ = +400.6769 ± 29.2953  (n_RR = 188)
r =    12.7 km  ξ = +377.3139 ± 24.4711  (n_RR = 239)
r =    14.8 km  ξ = +388.7789 ± 22.9282  (n_RR = 289)
r =    17.1 km  ξ = +299.9277 ± 14.2975  (n_RR = 443)
r =    19.8 km  ξ = +258.2750 ± 10.6114  (n_RR = 597)
r =    22.9 km  ξ = +219.4793 ± 7.7903  (n_RR = 801)
r =    26.5 km  ξ = +192.8156 ± 6.1107  (n_RR = 1006)
r =    30.7 km  ξ = +148.3290 ± 4.0597  (n_RR = 1353)
r =    35.5 km  ξ = +114.6171 ± 2.7829  (n_RR = 1726)
r =    41.1 km  ξ = +82.6249 ± 1.7081  (n_RR = 2397)
r =    47.6 km  ξ = +64.8839 ± 1.1670  (n_RR = 3187)
r =    55.2 km  ξ = +46.4374 ± 0.7242  (n_RR = 4291)
r =    63.9 km  ξ = +31.8095 ± 0.4270  (n_RR = 5904)
r =    73.9 km  ξ = +21.2763 ± 0.2531  (n_RR = 7748)
r =    85.6 km  ξ = +14.7309 ± 0.1554  (n_RR = 10243)
r =    99.1 km  ξ = +9.9995 ± 0.0941  (n_RR = 13666)
r =   114.8 km  ξ = +9.7050 ± 0.0793

In [9]:
// --- 可視化 ---
// エラーバー: geomLineRange（縦線）+ geomRibbon（帯）の二重表示
// Poisson 誤差 σ_ξ ≈ (1+ξ) / √n_RR を ±1σ で描画
val plotData = mapOf(
    "r"      to xiResult.map { it.rCenter },
    "xi"     to xiResult.map { it.xi },
    "xiLow"  to xiResult.map { it.xi - it.xiErr },
    "xiHigh" to xiResult.map { it.xi + it.xiErr }
)

letsPlot(plotData) +
    geomRibbon(alpha = 0.20, fill = "#4682B4") {
        x = "r"; ymin = "xiLow"; ymax = "xiHigh"
    } +
    geomLineRange(color = "#4682B4", size = 0.6, alpha = 0.8) {
        x = "r"; ymin = "xiLow"; ymax = "xiHigh"
    } +
    geomLine(color = "#4682B4", size = 1.0) { x = "r"; y = "xi" } +
    geomPoint(color = "#4682B4", size = 2.0) { x = "r"; y = "xi" } +
    geomHLine(yintercept = 0.0, linetype = "dashed", color = "#808080") +
    scaleXLog10(name = "分離距離 r [km]") +
    scaleYContinuous(name = "ξ(r)") +
    ggtitle(
        "Coles店舗の2点相関関数（Landy-Szalay推定量）",
        "エラーバー: Poisson 誤差 ±1σ ≈ (1+ξ)/√n_RR"
    ) +
    theme(plotBackground = elementRect(fill = "#ffffff"))

<path d="M0.0 28.259113168180534 L0.0 28.259113168180534 L14.305842150250356 13.727272727272748 L28.611684300500713 93.29015679527481 L42.91752645075107 106.05593136730175 L57.223368601001425 101.56229995365234 L71.52921075125181 145.71114484900343 L85.83505290150214 166.24475343426278 L100.14089505175252 185.0927232336238 L114.44673720200285 197.92918280839578 L128.75257935225324 219.00572132260106 L143.05842150250362 234.85180048584817 L157.364263652754 249.82763063500738 L171.67010580300433 258.1074351745087 L185.97594795325472 266.66228994042064 L200.28179010350505 273.42174645141455 L214.58763225375543 278.27091969271805 L228.89347440400576 281.2795204765715 L243.1993165542561 283.4500935644537 L257.5051587045064 283.59020059954855 L271.8110008547568 285.017444334858 L286.1168430050072 285.68871021765534 L300.42268515525757 286.75996802232277 L314.72852730550784 287.1395886826004 L329.0343694557583 286.64075666671164 L343.3402116060086 287.53534098880357 L357.64605375625894 287.7265172484324 L371.9518959065094 287.6432477768961 L386.2577380567597 287.5573988969798 L400.56358020701003 287.8073825302634 L414.8694223572605 287.67374922923796 L429.1752645075108 285.8259027276952 L443.481106657761 287.14444950412854 L457.7869488080116 287.9825464134524 L472.0927909582619 288.1165300947512 L486.39863310851223 287.975786528756 L500.70447525876256 287.7466067153636 L515.0103174090129 288.1396833111862 L529.3161595592633 288.2724599173372 L529.3161595592633 288.27272727272725 L515.0103174090129 288.14016302727157 L500.70447525876256 287.7477515090258 L486.39863310851223 287.9766556250759 L472.0927909582619 288.11722813106667 L457.7869488080116 287.98362569513813 L443.481106657761 287.1477483334536 L429.1752645075108 285.8333441960926 L414.8694223572605 287.67631204886635 L400.56358020701003 287.809815493042 L386.2577380567597 287.5612258580047 L371.9518959065094 287.64722524828545 L357.64605375625894 287.73062269744946 L343.3402116060086 287.5412591968287 L329.0343694557583 286.6539995654464 L314.72852730550784 287.15067797091285 L300.42268515525757 286.7763550540221 L286.1168430050072 285.7193772467871 L271.8110008547568 285.06141999169387 L257.5051587045064 283.662016140135 L243.1993165542561 283.5353210050808 L228.89347440400576 281.4203083289148 L214.58763225375543 278.50015110875523 L200.28179010350505 273.80851574875186 L185.97594795325472 267.31823444299033 L171.67010580300433 259.1645292124494 L157.364263652754 251.3747617911722 L143.05842150250362 237.37253500098177 L128.75257935225324 222.68294802535513 L114.44673720200285 203.4641491726593 L100.14089505175252 192.14901964653856 L85.83505290150214 175.8564205514059 L71.52921075125181 158.66161591090685 L57.223368601001425 122.33030243352835 L42.91752645075107 128.22149015103517 L28.611684300500713 119.82541111059021 L14.305842150250356 60.597216851639644 L0.0 77.63866158131762 Z" fill="rgb(70,130,180)" stroke-width="1.0" fill-opacity="0.2">
 
 
 
 <path d="M0.0 28.259113168180534 L0.0 28.259113168180534 L14.305842150250356 13.727272727272748 L28.611684300500713 93.29015679527481 L42.91752645075107 106.05593136730175 L57.223368601001425 101.56229995365234 L71.52921075125181 145.71114484900343 L85.83505290150214 166.24475343426278 L100.14089505175252 185.0927232336238 L114.44673720200285 197.92918280839578 L128.75257935225324 219.00572132260106 L143.05842150250362 234.85180048584817 L157.364263652754 249.82763063500738 L171.67010580300433 258.1074351745087 L185.97594795325472 266.66228994042064 L200.28179010350505 273.42174645141455 L214.58763225375543 278.27091969271805 L228.89347440400576 281.2795204765715 L243.1993165542561 283.4500935644537 L257.5051587045064 283.59020059954855 L271.8110008547568 285.017444334858 L286.1168430050072 285.68871021765534 L300.42268515525757 286.75996802232277 L314.72852730550784 287.1395886826004 L329.0343694557583 286.64075666671164 L343.3402116060086 287.53534098880357 L357.64605375625894 287.7265172484324 L371.9518959065094 287.64324777

## 大陸マスク付きランダムカタログによる比較検証

現在のランダムカタログはバウンディングボックス内の一様サンプリング（海洋点 ~45% を含む）。
この幾何学的バイアスが特に中〜大スケール（$r > 300$ km）の $\xi(r)$ に系統誤差を与えている可能性がある。

とくに $r \approx 725$ km のバンプが「Sydney–Melbourne / Melbourne–Adelaide の実シグナル」か
「矩形カタログの幾何アーティファクト」かを判定するため、
オーストラリア本土ポリゴンでマスクした random catalog と比較する。

### テスト手順
1. オーストラリア大陸の簡略化ポリゴン（~37頂点）を定義し、Ray-casting 法で陸上判定
2. マスク済みランダムカタログ（$N_R^{\rm masked} = 10\,N_D$）を生成
3. $DD$ は共通、$DR^{\rm masked}$ と $RR^{\rm masked}$ のみ再計算
4. $\hat{\xi}^{\rm masked}(r)$ を元の $\hat{\xi}(r)$ と重ねてプロット

In [10]:
// --- Natural Earth ベースの高精度ポリゴン（2ポリゴン）を CSV から読み込む ---
// australia_mask_polygon.csv: lat, lon, polygon_id (0=本土, 1=Tasmania)
// 生成: Natural Earth ne_10m_admin_0_countries + shapely tol=0.02 + buffer(0.02 deg ≈ 2 km)
// 精度: 685/685 点を陸上判定（旧37頂点ポリゴンは 650/685 = 94.9%）

val polyDf = DataFrame.readCSV("./output/australia_mask_polygon.csv")

// polygon_id ごとに頂点リストを分離
val mainland: List<Pair<Double, Double>> = polyDf
    .filter { (it["polygon_id"] as Int) == 0 }
    .rows().map { Pair(it["lat"] as Double, it["lon"] as Double) }

val tasmania: List<Pair<Double, Double>> = polyDf
    .filter { (it["polygon_id"] as Int) == 1 }
    .rows().map { Pair(it["lat"] as Double, it["lon"] as Double) }

println("本土ポリゴン: ${mainland.size} 頂点")
println("Tasmania ポリゴン: ${tasmania.size} 頂点")

// --- Ray-casting 法によるポリゴン内判定 ---
fun isInsidePolygon(lat: Double, lon: Double, poly: List<Pair<Double, Double>>): Boolean {
    var inside = false
    var j = poly.size - 1
    for (i in poly.indices) {
        val (latI, lonI) = poly[i]
        val (latJ, lonJ) = poly[j]
        if ((latI > lat) != (latJ > lat)) {
            val lonCross = (lonJ - lonI) * (lat - latI) / (latJ - latI) + lonI
            if (lon < lonCross) inside = !inside
        }
        j = i
    }
    return inside
}

// 本土 or Tasmania のいずれかに含まれれば陸上
// （旧: isTasmania() の矩形補助判定は不要になった）
fun isOnContinent(lat: Double, lon: Double): Boolean =
    isInsidePolygon(lat, lon, mainland) || isInsidePolygon(lat, lon, tasmania)

// --- 陸上判定率の検証 ---
val dataOnLand = dataPoints.count { isOnContinent(it.lat, it.lon) }
val offshorePoints = dataPoints.filter { !isOnContinent(it.lat, it.lon) }
println("\n--- 陸上判定 ---")
println("陸上: $dataOnLand / $nD  (${"%.1f".format(dataOnLand.toDouble() / nD * 100)} %)")
println("海洋判定（要確認）: ${offshorePoints.size} 点")
if (offshorePoints.isNotEmpty()) {
    println("海洋判定点の座標:")
    offshorePoints.sortedBy { it.lat }.forEach {
        println("  lat=${ "%.4f".format(it.lat) }, lon=${ "%.4f".format(it.lon) }")
    }
}

本土ポリゴン: 1751 頂点
Tasmania ポリゴン: 193 頂点

--- 陸上判定 ---
陸上: 683 / 685  (99.7 %)
海洋判定（要確認）: 2 点
海洋判定点の座標:
  lat=-38.2192, lon=145.0399
  lat=-16.9239, lon=145.7709


In [11]:
// --- マスク済みランダムカタログの生成 (rejection sampling) ---
val rngMasked = Random(42)
val randomMasked: List<Point> = generateSequence {
    Point(
        lat = latMin + rngMasked.nextDouble() * (latMax - latMin),
        lon = lonMin + rngMasked.nextDouble() * (lonMax - lonMin)
    )
}.filter { isOnContinent(it.lat, it.lon) }
 .take(nR)
 .toList()

// f_land を推定（acceptance rate）
val rngEst = Random(99)
val trials = 100_000
val accepted = (0 until trials).count {
    isOnContinent(
        latMin + rngEst.nextDouble() * (latMax - latMin),
        lonMin + rngEst.nextDouble() * (lonMax - lonMin)
    )
}
val fLandEst = accepted.toDouble() / trials
println("マスク済みカタログ: N_R = ${randomMasked.size}")
println("f_land 推定値 = ${"%.3f".format(fLandEst)}  （海洋点の割合 ≈ ${"%.1f".format((1 - fLandEst) * 100)} %）")

マスク済みカタログ: N_R = 6850
f_land 推定値 = 0.502  （海洋点の割合 ≈ 49.8 %）


### マスク検証プロット

ランダムサンプル 3000 点を陸海に色分けし、ポリゴン境界と Coles 店舗を重ねて描画する。
- **緑**: 陸上（採用） — マスク済みカタログに使われる点
- **青**: 海洋（除外） — 矩形カタログにのみ含まれる点
- **赤**: Coles 店舗
- **黒線**: マスクポリゴン境界

店舗が黒線の内側に収まっていれば、ポリゴンの精度が確認できる。

In [12]:
import org.jetbrains.letsPlot.coord.*

// ランダムサンプルを生成して陸海判定
val rngViz = Random(777)
val vizN = 3000
val vizLats = List(vizN) { latMin + rngViz.nextDouble() * (latMax - latMin) }
val vizLons = List(vizN) { lonMin + rngViz.nextDouble() * (lonMax - lonMin) }
val vizKind = (vizLats zip vizLons).map { (la, lo) ->
    if (isOnContinent(la, lo)) "陸上（採用）" else "海洋（除外）"
}
val maskVizData = mapOf("lat" to vizLats, "lon" to vizLons, "kind" to vizKind)

// 2ポリゴン境界（本土 + Tasmania）を連結してプロット
// polygon_id を group に使い、2本の境界線を別パスとして描画
val polyLats   = mainland.map { it.first }  + tasmania.map { it.first }
val polyLons   = mainland.map { it.second } + tasmania.map { it.second }
val polyGroup  = List(mainland.size) { "本土" } + List(tasmania.size) { "Tasmania" }
val polyData = mapOf("lat" to polyLats, "lon" to polyLons, "group" to polyGroup)

// Coles 店舗
val storeData = mapOf(
    "lat" to dataPoints.map { it.lat },
    "lon" to dataPoints.map { it.lon }
)

letsPlot() +
    geomPoint(data = maskVizData, alpha = 0.35, size = 0.8) {
        x = "lon"; y = "lat"; color = "kind"
    } +
    geomPath(data = polyData, color = "#111111", size = 0.8) {
        x = "lon"; y = "lat"; group = "group"
    } +
    geomPoint(data = storeData, color = "#CC2222", size = 1.5, alpha = 0.8) {
        x = "lon"; y = "lat"
    } +
    scaleColorManual(
        values = mapOf("陸上（採用）" to "#2CA02C", "海洋（除外）" to "#4682B4")
    ) +
    coordMap(xlim = Pair(110.0, 155.0), ylim = Pair(-45.0, -10.0)) +
    ggtitle(
        "大陸マスク検証（Natural Earth ne_10m + buffer 2 km）",
        "赤=Coles店舗（685点）、黒線=マスク境界。全店舗が黒線内に収まることを確認"
    ) +
    theme(plotBackground = elementRect(fill = "#ffffff"))

In [13]:
// --- マスク済みカタログで DR・RR を再計算 ---
// DD は変わらないため nDD を再利用
val nRMasked = randomMasked.size

println("DR_masked を計算中...")
val nDR_masked = pairCounts(dataPoints, randomMasked, bins)

println("RR_masked を計算中...")
val nRR_masked = pairCounts(randomMasked, null, bins)

println("完了")

// --- マスク済み Landy-Szalay 推定量 ---
val normDR_masked = nD.toLong() * nRMasked
val normRR_masked = nRMasked.toLong() * (nRMasked - 1) / 2

val xiMasked: List<XiBin> = (0 until nBins).map { i ->
    val dd = nDD[i].toDouble() / normDD
    val dr = nDR_masked[i].toDouble() / normDR_masked
    val rr = nRR_masked[i].toDouble() / normRR_masked
    val xi = if (rr > 0.0) (dd - 2.0 * dr + rr) / rr else Double.NaN
    val xiErr = if (nRR_masked[i] > 0) (1.0 + xi) / sqrt(nRR_masked[i].toDouble()) else Double.NaN
    XiBin(rCenters[i], xi, xiErr, nRR_masked[i])
}

println("\n--- マスクあり vs マスクなし ---")
println("%-10s  %-12s  %-12s  %-10s".format("r [km]", "ξ (矩形)", "ξ (マスク)", "Δξ"))
xiResult.zip(xiMasked).forEach { (orig, masked) ->
    val delta = masked.xi - orig.xi
    println("%-10.1f  %-12.4f  %-12.4f  %-10.4f".format(
        orig.rCenter, orig.xi, masked.xi, delta))
}

DR_masked を計算中...
RR_masked を計算中...
完了

--- マスクあり vs マスクなし ---
r [km]      ξ (矩形)        ξ (マスク)       Δξ        
8.2         519.0469      229.5966      -289.4503 
9.5         553.9043      263.1330      -290.7713 
11.0        400.6769      217.6067      -183.0702 
12.7        377.3139      185.7089      -191.6050 
14.8        388.7789      184.4402      -204.3387 
17.1        299.9277      159.9554      -139.9723 
19.8        258.2750      144.5967      -113.6783 
22.9        219.4793      118.7438      -100.7355 
26.5        192.8156      98.4448       -94.3708  
30.7        148.3290      78.6296       -69.6993  
35.5        114.6171      55.4220       -59.1952  
41.1        82.6249       43.0323       -39.5926  
47.6        64.8839       33.4794       -31.4045  
55.2        46.4374       24.3087       -22.1288  
63.9        31.8095       17.0284       -14.7812  
73.9        21.2763       11.2160       -10.0603  
85.6        14.7309       7.7462        -6.9847   
99.1        9.9995 

In [14]:
// --- ξ_CC(r) を CSV に出力 ---
// 出力: output/xi_cc_masked.csv
run {
    val file = java.io.File("./output/xi_cc_masked.csv")
    file.bufferedWriter().use { w ->
        w.appendLine("r_km,xi_cc,xi_err,n_rr")
        xiMasked.forEach { b ->
            w.appendLine("${b.rCenter},${b.xi},${b.xiErr},${b.nRR}")
        }
    }
    println("保存: ${file.path}  (${xiMasked.size} 行)")
}


保存: ./output/xi_cc_masked.csv  (38 行)


In [15]:
// --- 比較プロット: 矩形カタログ vs 大陸マスク済みカタログ ---
// エラーバー: geomLineRange（縦線）+ geomRibbon（帯）
val compData = mapOf(
    "r"       to (xiResult.map { it.rCenter } + xiMasked.map { it.rCenter }),
    "xi"      to (xiResult.map { it.xi }      + xiMasked.map { it.xi }),
    "xiLow"   to (xiResult.map { it.xi - it.xiErr } + xiMasked.map { it.xi - it.xiErr }),
    "xiHigh"  to (xiResult.map { it.xi + it.xiErr } + xiMasked.map { it.xi + it.xiErr }),
    "catalog" to (List(nBins) { "矩形（海洋含む）" } + List(nBins) { "大陸マスク済み" })
)

letsPlot(compData) +
    geomRibbon(alpha = 0.12) {
        x = "r"; ymin = "xiLow"; ymax = "xiHigh"; fill = "catalog"
    } +
    geomLineRange(size = 0.5, alpha = 0.7) {
        x = "r"; ymin = "xiLow"; ymax = "xiHigh"; color = "catalog"
    } +
    geomLine(size = 1.2) { x = "r"; y = "xi"; color = "catalog" } +
    geomPoint(size = 2.0) { x = "r"; y = "xi"; color = "catalog" } +
    geomHLine(yintercept = 0.0, linetype = "dashed", color = "#808080") +
    geomVLine(xintercept = 725.0, linetype = "dotted", color = "#CC4444") +
    scaleXLog10(name = "分離距離 r [km]") +
    scaleYContinuous(name = "ξ(r)") +
    scaleColorManual(values = mapOf("矩形（海洋含む）" to "#4682B4", "大陸マスク済み" to "#E87040")) +
    scaleFillManual(values = mapOf("矩形（海洋含む）" to "#4682B4", "大陸マスク済み" to "#E87040")) +
    ggtitle(
        "ξ(r) 比較：ランダムカタログの幾何バイアス検証",
        "エラーバー: Poisson ±1σ。赤点線 = r=725 km（Melb–Adel）"
    ) +
    theme(plotBackground = elementRect(fill = "#ffffff"))

<path d="M0.0 28.259113168180534 L0.0 28.259113168180534 L10.89956039093164 13.727272727272748 L21.79912078186328 93.29015679527481 L32.69868117279492 106.05593136730175 L43.59824156372656 101.56229995365234 L54.49780195465823 145.71114484900343 L65.39736234558984 166.24475343426278 L76.29692273652151 185.0927232336238 L87.19648312745312 197.92918280839578 L98.09604351838479 219.00572132260106 L108.99560390931643 234.85180048584817 L119.89516430024807 249.82763063500738 L130.7947246911797 258.1074351745087 L141.69428508211135 266.66228994042064 L152.593845473043 273.42174645141455 L163.49340586397463 278.27091969271805 L174.39296625490627 281.2795204765715 L185.2925266458379 283.4500935644537 L196.19208703676955 283.59020059954855 L207.0916474277012 285.017444334858 L217.99120781863283 285.68871021765534 L228.89076820956453 286.75996802232277 L239.7903286004961 287.1395886826004 L250.6898889914278 286.64075666671164 L261.58944938235936 287.53534098880357 L272.48900977329106 287.7265172484324 L283.38857016422276 287.6432477768961 L294.28813055515434 287.5573988969798 L305.1876909460859 287.8073825302634 L316.08725133701773 287.67374922923796 L326.9868117279492 285.8259027276952 L337.8863721188808 287.14444950412854 L348.7859325098126 287.9825464134524 L359.6854929007442 288.1165300947512 L370.58505329167576 287.975786528756 L381.48461368260746 287.7466067153636 L392.38417407353904 288.1396833111862 L403.28373446447074 288.2724599173372 L403.28373446447074 288.27272727272725 L392.38417407353904 288.14016302727157 L381.48461368260746 287.7477515090258 L370.58505329167576 287.9766556250759 L359.6854929007442 288.11722813106667 L348.7859325098126 287.98362569513813 L337.8863721188808 287.1477483334536 L326.9868117279492 285.8333441960926 L316.08725133701773 287.67631204886635 L305.1876909460859 287.809815493042 L294.28813055515434 287.5612258580047 L283.38857016422276 287.64722524828545 L272.48900977329106 287.73062269744946 L261.58944938235936 287.5412591968287 L250.6898889914278 286.6539995654464 L239.7903286004961 287.15067797091285 L228.89076820956453 286.7763550540221 L217.99120781863283 285.7193772467871 L207.0916474277012 285.06141999169387 L196.19208703676955 283.662016140135 L185.2925266458379 283.5353210050808 L174.39296625490627 281.4203083289148 L163.49340586397463 278.50015110875523 L152.593845473043 273.80851574875186 L141.69428508211135 267.31823444299033 L130.7947246911797 259.1645292124494 L119.89516430024807 251.3747617911722 L108.99560390931643 237.37253500098177 L98.09604351838479 222.68294802535513 L87.19648312745312 203.4641491726593 L76.29692273652151 192.14901964653856 L65.39736234558984 175.8564205514059 L54.49780195465823 158.66161591090685 L43.59824156372656 122.33030243352835 L32.69868117279492 128.22149015103517 L21.79912078186328 119.82541111059021 L10.89956039093164 60.597216851639644 L0.0 77.63866158131762 Z" fill="rgb(70,130,180)" stroke-width="1.0" fill-opacity="0.12156862745098039">
 
 
 
 <path d="M0.0 176.7447035803886 L0.0 176.7447035803886 L10.89956039093164 161.14477356758215 L21.79912078186328 184.1386641100445 L32.69868117279492 200.071631525498 L43.59824156372656 201.08097735379374 L54.49780195465823 213.04550090253582 L65.39736234558984 220.512180214797 L76.29692273652151 232.83117748879965 L87.19648312745312 242.41971039951454 L98.09604351838479 251.69464297487164 L108.99560390931643 262.4922861770946 L119.89516430024807 268.2372174341824 L130.7947246911797 272.65929144273434 L141.69428508211135 276.8850989545131 L152.593845473043 280.23112023085224 L163.49340586397463 282.89568332520435 L174.39296625490627 284.4847812733271 L185.2925266458379 285.6062268292493 L196.19208703676955 285.6243917560326 L207.0916474277012 286.37631553495925 L217.99120781863283 286.75053082412785 L228.89076820956453 287.27841846662585 L239.7903286004961 287.4950226102814 L250.6898889914278 287.18461543488667 L261.58944938235936 287.69543418650136 L272.48900977329106 287.82182951771625 L283.38857016422276 287.7

## 交差相関関数 $\xi_{CW}(r)$：Coles–Woolworths duopoly の空間的排斥

Coles 点過程 $C$ と Woolworths 点過程 $W$ の交差2点相関関数を計算する。

$$\hat{\xi}_{CW}(r) = \frac{D_C D_W(r) - D_C R_W(r) - D_W R_C(r) + R_C R_W(r)}{R_C R_W(r)}$$

ここで $D_C D_W$ は Coles–Woolworths 異種ペアカウント、$D R$ / $R R$ は各ランダムカタログとの混合ペアカウント。

- $\xi_{CW}(r) > 0$：スケール $r$ で Coles–Woolworths が共存する傾向（同一商圏内に双方が出店）
- $\xi_{CW}(r) < 0$：スケール $r$ で空間的排斥（競合回避、市場分割）
- 自己相関 $\xi_{CC}(r)$ との比較により、競合他社存在下でのクラスタリング特性の変化を定量化する。

In [16]:
// --- Woolworths 店舗データ読み込み ---
// 事前に woolworths_location.ipynb を実行して woolworths_locations.csv を生成しておくこと
val dfW = DataFrame.readCSV("./output/woolworths_locations.csv")

data class PointW(val lat: Double, val lon: Double)

val woolworthsPoints: List<PointW> = dfW.rows().map { row ->
    PointW(
        lat = row["lat"].toString().toDouble(),
        lon = row["lon"].toString().toDouble()
    )
}

val nW = woolworthsPoints.size
println("Woolworths 店舗数: $nW")
println("Coles 店舗数:      $nD")

Woolworths 店舗数: 1039
Coles 店舗数:      685


In [17]:
// ⚠️ 前提: Cell 11 (isOnContinent 定義) と Cell 12 (randomMasked 生成) が実行済みであること

// --- 汎用ハーバーサイン距離関数（PointW 対応） ---
fun haversineW(lat1: Double, lon1: Double, lat2: Double, lon2: Double): Double {
    val R = 6371.0
    val dLat = Math.toRadians(lat2 - lat1)
    val dLon = Math.toRadians(lon2 - lon1)
    val a = sin(dLat / 2).pow(2) +
            cos(Math.toRadians(lat1)) * cos(Math.toRadians(lat2)) * sin(dLon / 2).pow(2)
    return 2 * R * asin(sqrt(a))
}

// --- Woolworths 用大陸マスク済みランダムカタログ ---
// R_C R_W のペア数 = N_RC × N_RW。N_RW = 10×N_W だと ~1億ペアになるため
// N_RW = 5000 に固定してシェットノイズを許容範囲内に抑える。
// 誤差への影響: σ ∝ 1/√N_RW → 5000 は 10×N_W に対して √2 倍程度の増加にとどまる。
val nRW_target = 5000
val rngW = kotlin.random.Random(123)
val randomW: List<PointW> = generateSequence {
    val lat = rngW.nextDouble(-44.0, -10.0)
    val lon = rngW.nextDouble(113.0, 154.0)
    PointW(lat, lon)
}.filter { isOnContinent(it.lat, it.lon) }
 .take(nRW_target)
 .toList()

val nRW = randomW.size
println("Woolworths ランダムカタログ: $nRW 点  (R_C×R_W ペア数: ${nRMasked.toLong() * nRW / 1_000_000}M)")


Woolworths ランダムカタログ: 5000 点  (R_C×R_W ペア数: 34M)


In [18]:
// --- 交差ペアカウント関数 ---
// 異種2点間距離カウント（i < j の自己除外なし。全ペアをカウント）
fun crossPairCounts(
    points1: List<Point>,
    points2: List<PointW>,
    bins: DoubleArray
): LongArray {
    val counts = LongArray(bins.size - 1)
    for (p1 in points1) {
        for (p2 in points2) {
            val d = haversineW(p1.lat, p1.lon, p2.lat, p2.lon)
            val idx = bins.indexOfFirst { d < it } - 1
            if (idx in counts.indices) counts[idx]++
        }
    }
    return counts
}

// D_C D_W
println("D_C D_W を計算中...")
val nDcDw = crossPairCounts(dataPoints, woolworthsPoints, bins)

// D_C R_W
println("D_C R_W を計算中...")
val nDcRw = crossPairCounts(dataPoints, randomW, bins)

// D_W R_C  (Coles の大陸マスク済みランダムカタログ randomMasked を流用)
println("D_W R_C を計算中...")
val nDwRc = run {
    val counts = LongArray(bins.size - 1)
    for (p1 in woolworthsPoints) {
        for (p2 in randomMasked) {
            val d = haversineW(p1.lat, p1.lon, p2.lat, p2.lon)
            val idx = bins.indexOfFirst { d < it } - 1
            if (idx in counts.indices) counts[idx]++
        }
    }
    counts
}

// R_C R_W
println("R_C R_W を計算中...")
val nRcRw = crossPairCounts(randomMasked, randomW, bins)

println("完了")

D_C D_W を計算中...
D_C R_W を計算中...
D_W R_C を計算中...
R_C R_W を計算中...
完了


In [19]:
// --- 交差相関関数 ξ_CW(r) ---
// 正規化係数
val normDcDw = nD.toLong() * nW                    // D_C × D_W
val normDcRw = nD.toLong() * nRW                   // D_C × R_W
val normDwRc = nW.toLong() * nRMasked              // D_W × R_C
val normRcRw = nRMasked.toLong() * nRW             // R_C × R_W

data class XiBinCW(val rCenter: Double, val xi: Double, val xiErr: Double, val nRR: Long)

val xiCW: List<XiBinCW> = (0 until nBins).map { i ->
    val dcDw = nDcDw[i].toDouble() / normDcDw
    val dcRw = nDcRw[i].toDouble() / normDcRw
    val dwRc = nDwRc[i].toDouble() / normDwRc
    val rcRw = nRcRw[i].toDouble() / normRcRw
    val xi   = if (rcRw > 0.0) (dcDw - dcRw - dwRc + rcRw) / rcRw else Double.NaN
    val xiErr = if (nRcRw[i] > 0) (1.0 + xi) / sqrt(nRcRw[i].toDouble()) else Double.NaN
    XiBinCW(rCenters[i], xi, xiErr, nRcRw[i])
}

println("%-10s  %-12s  %-12s  %-12s".format("r [km]", "ξ_CC (Coles)", "ξ_CW (cross)", "ξ_CC - ξ_CW"))
xiMasked.zip(xiCW).forEach { (cc, cw) ->
    val diff = cc.xi - cw.xi
    println("%-10.1f  %-12.4f  %-12.4f  %-12.4f".format(cc.rCenter, cc.xi, cw.xi, diff))
}

r [km]      ξ_CC (Coles)  ξ_CW (cross)  ξ_CC - ξ_CW 
8.2         229.5966      230.1173      -0.5206     
9.5         263.1330      232.2466      30.8864     
11.0        217.6067      207.1814      10.4254     
12.7        185.7089      174.8560      10.8529     
14.8        184.4402      169.0676      15.3727     
17.1        159.9554      150.4283      9.5271      
19.8        144.5967      134.3201      10.2766     
22.9        118.7438      112.6759      6.0679      
26.5        98.4448       90.9942       7.4507      
30.7        78.6296       71.8487       6.7810      
35.5        55.4220       53.2014       2.2206      
41.1        43.0323       38.5237       4.5086      
47.6        33.4794       29.4276       4.0518      
55.2        24.3087       21.5464       2.7623      
63.9        17.0284       15.9141       1.1143      
73.9        11.2160       11.6187       -0.4026     
85.6        7.7462        8.0125        -0.2663     
99.1        5.2936        5.4653        -0.171

In [20]:
// --- ξ_CC / ξ_CW 広域 (対数ビン) を CSV に出力 ---
// 出力: output/xi_cw_wide.csv
run {
    val file = java.io.File("./output/xi_cw_wide.csv")
    file.bufferedWriter().use { w ->
        w.appendLine("r_km,xi_cc,xi_cc_err,xi_cw,xi_cw_err")
        xiMasked.zip(xiCW).forEach { (cc, cw) ->
            w.appendLine("${cc.rCenter},${cc.xi},${cc.xiErr},${cw.xi},${cw.xiErr}")
        }
    }
    println("保存: ${file.path}  (${xiMasked.size} 行)")
}


保存: ./output/xi_cw_wide.csv  (38 行)


In [21]:
// --- 可視化: ξ_CC(r) vs ξ_CW(r) ---
// ξ_CC > ξ_CW → 同種クラスタリングが異種より強い（競合回避）
// ξ_CW < 0    → スケール r で duopoly が互いを排除（市場分割）

val cwPlotData = mapOf(
    "r"      to (xiMasked.map { it.rCenter } + xiCW.map { it.rCenter }),
    "xi"     to (xiMasked.map { it.xi }      + xiCW.map { it.xi }),
    "xiLow"  to (xiMasked.map { it.xi - it.xiErr } + xiCW.map { it.xi - it.xiErr }),
    "xiHigh" to (xiMasked.map { it.xi + it.xiErr } + xiCW.map { it.xi + it.xiErr }),
    "label"  to (List(nBins) { "ξ_CC  Coles 自己相関" } + List(nBins) { "ξ_CW  Coles×Woolworths 交差相関" })
)

letsPlot(cwPlotData) +
    geomRibbon(alpha = 0.12) {
        x = "r"; ymin = "xiLow"; ymax = "xiHigh"; fill = "label"
    } +
    geomLineRange(size = 0.5, alpha = 0.7) {
        x = "r"; ymin = "xiLow"; ymax = "xiHigh"; color = "label"
    } +
    geomLine(size = 1.2) { x = "r"; y = "xi"; color = "label" } +
    geomPoint(size = 2.0) { x = "r"; y = "xi"; color = "label" } +
    geomHLine(yintercept = 0.0, linetype = "dashed", color = "#808080") +
    geomVLine(xintercept = 725.0, linetype = "dotted", color = "#CC4444") +
    scaleXLog10(name = "分離距離 r [km]") +
    scaleYContinuous(name = "ξ(r)") +
    scaleColorManual(values = mapOf(
        "ξ_CC  Coles 自己相関"             to "#4682B4",
        "ξ_CW  Coles×Woolworths 交差相関"   to "#E87040"
    )) +
    scaleFillManual(values = mapOf(
        "ξ_CC  Coles 自己相関"             to "#4682B4",
        "ξ_CW  Coles×Woolworths 交差相関"   to "#E87040"
    )) +
    ggtitle(
        "ξ_CC(r) vs ξ_CW(r)：Coles–Woolworths duopoly の空間的排斥",
        "ξ_CW < 0 → 市場分割（競合回避）。赤点線 = r=725 km (Retail BAO)"
    ) +
    theme(plotBackground = elementRect(fill = "#ffffff"))

<path d="M0.0 47.41952426334433 L0.0 47.41952426334433 L6.962627563918787 13.727272727272748 L13.925255127837588 63.38877398487773 L20.887882691756374 97.80030951335777 L27.850510255675175 99.98026375071063 L34.81313781959399 125.8208756369526 L41.77576538351276 141.94718116020593 L48.738392947431564 168.5533745525574 L55.701020511350336 189.2623945561405 L62.66364807526914 209.29411015053745 L69.62627563918794 232.61452956699623 L76.58890320310674 245.02225632040745 L83.55153076702554 254.57291646343026 L90.51415833094434 263.6996862318195 L97.47678589486311 270.92632060741727 L104.43941345878191 276.68116245189634 L111.40204102270069 280.1132475791792 L118.36466858661949 282.5353113783836 L125.32729615053829 282.57454343146424 L132.28992371445707 284.1985254028419 L139.2525512783759 285.0067441346542 L146.21517884229468 286.14685971332835 L153.17780640621345 286.61467471543455 L160.14043397013228 285.9442667954365 L167.10306153405105 287.0475175058317 L174.06568909796982 287.32050227491607 L181.02831666188865 287.18781861595454 L187.99094422580737 287.1357873812346 L194.9535717897262 287.33905207606296 L201.91619935364503 287.02773546314654 L208.8788269175638 284.5870782215286 L215.84145448148257 286.29623813625903 L222.8040820454014 287.51542632681753 L229.76670960932023 287.7808814535349 L236.72933717323895 287.5579154548872 L243.69196473715778 287.268851837432 L250.6545923010766 287.933044260293 L257.6172198649954 288.2647988817933 L257.6172198649954 288.2653915123441 L250.6545923010766 287.93408121126384 L243.69196473715778 287.270839013732 L236.72933717323895 287.55962090877597 L229.76670960932023 287.7823875460122 L222.8040820454014 287.5175526482709 L215.84145448148257 286.30098081185196 L208.8788269175638 284.5961504871501 L201.91619935364503 287.03193798852243 L194.9535717897262 287.3429386352649 L187.99094422580737 287.14084826411283 L181.02831666188865 287.1933936921215 L174.06568909796982 287.3263127566433 L167.10306153405105 287.0554477256134 L160.14043397013228 285.9592895896086 L153.17780640621345 286.62773732985255 L146.21517884229468 286.1651559492811 L139.2525512783759 285.0370284460887 L132.28992371445707 284.24088153084557 L125.32729615053829 282.6409127557538 L118.36466858661949 282.612014014225 L111.40204102270069 280.2361792921004 L104.43941345878191 276.8802575910568 L97.47678589486311 271.26449009164867 L90.51415833094434 264.24875906283484 L83.55153076702554 255.4349920773382 L76.58890320310674 246.2973189881857 L69.62627563918794 234.46732728831586 L62.66364807526914 212.38687742672886 L55.701020511350336 193.65442490358635 L48.738392947431564 174.65281062212267 L41.77576538351276 150.68328099346658 L34.81313781959399 136.76354028481674 L27.850510255675175 114.70482188512185 L20.887882691756374 114.4028683524296 L13.925255127837588 86.41308586813005 L6.962627563918787 47.01213790092092 L0.0 78.92664409488071 Z" fill="rgb(70,130,180)" stroke-width="1.0" fill-opacity="0.12156862745098039">
 
 
 
 <path d="M0.0 49.08083417847615 L0.0 49.08083417847615 L6.962627563918787 48.2621916806832 L13.925255127837588 75.93662225421548 L20.887882691756374 110.220422789878 L27.850510255675175 116.75874670707796 L34.81313781959399 136.3739061598238 L41.77576538351276 153.01497972269155 L48.738392947431564 175.10380606574074 L55.701020511350336 197.0589101866949 L62.66364807526914 216.31055482129796 L69.62627563918794 234.97143519957095 L76.58890320310674 249.60306042952146 L83.55153076702554 258.6535602996279 L90.51415833094434 266.4745787972969 L97.47678589486311 272.0548172923848 L104.43941345878191 276.30212093304726 L111.40204102270069 279.8618162658471 L118.36466858661949 282.3731892586273 L125.32729615053829 282.73398215654623 L132.28992371445707 283.9244338519028 L139.2525512783759 284.9254174879077 L146.21517884229468 286.0735939864943 L153.17780640621345 286.464460898007 L160.14043397013228 285.8972087158213 L167.10306153405105 286.8963539056751 L174.06568909796982 287.16719689026957 L181.02831666188865 28

## 小スケール交差相関 $\xi_{CW}(r)$：anchor clustering vs suburban 競合圏

対数ビン設計（$r_{\rm min} \approx 27$ km）では $r < 27$ km の構造が分解できない。
以下の2成分を分離するため、$\Delta r = 1$ km の線形ビンで $r = 0.5$–$30$ km を再計算する：

| 成分 | スケール | 予測 |
|---|---|---|
| Anchor clustering | $r < 1$ km | $\xi_{CW} \gg 0$（Westfield 等で Coles・Woolworths が同居）|
| Suburban 競合圏 | $r \sim 1$–$20$ km | $\xi_{CW}$ の符号変化（競合回避距離を定義する）|


In [22]:
// --- 小スケール専用線形ビン設計 ---
// Δr = 1 km, r = 0.5 ~ 30.5 km → 30 ビン
val rMinSmall  = 0.5
val rMaxSmall  = 30.5
val drSmall    = 1.0
val nBinsSmall = ((rMaxSmall - rMinSmall) / drSmall).toInt()  // 30

val binsSmall: DoubleArray = DoubleArray(nBinsSmall + 1) { i -> rMinSmall + i * drSmall }
val rCentersSmall: List<Double> = (0 until nBinsSmall).map { i ->
    (binsSmall[i] + binsSmall[i + 1]) / 2.0
}

println("小スケールビン: ${nBinsSmall} 本, Δr = ${drSmall} km, r = ${rMinSmall}–${rMaxSmall} km")


小スケールビン: 30 本, Δr = 1.0 km, r = 0.5–30.5 km


In [23]:
// --- 小スケール ξ_CW の計算 ---
println("D_C D_W (small) を計算中...")
val nDcDwSm = crossPairCounts(dataPoints, woolworthsPoints, binsSmall)

println("D_C R_W (small) を計算中...")
val nDcRwSm = crossPairCounts(dataPoints, randomW, binsSmall)

println("D_W R_C (small) を計算中...")
val nDwRcSm = run {
    val counts = LongArray(binsSmall.size - 1)
    for (p1 in woolworthsPoints) {
        for (p2 in randomMasked) {
            val d = haversineW(p1.lat, p1.lon, p2.lat, p2.lon)
            val idx = binsSmall.indexOfFirst { d < it } - 1
            if (idx in counts.indices) counts[idx]++
        }
    }
    counts
}

println("R_C R_W (small) を計算中...")
val nRcRwSm = crossPairCounts(randomMasked, randomW, binsSmall)

println("完了")

// --- ξ_CW 推定 ---
val xiCWSmall: List<XiBinCW> = (0 until nBinsSmall).map { i ->
    val dcDw = nDcDwSm[i].toDouble() / normDcDw
    val dcRw = nDcRwSm[i].toDouble() / normDcRw
    val dwRc = nDwRcSm[i].toDouble() / normDwRc
    val rcRw = nRcRwSm[i].toDouble() / normRcRw
    val xi    = if (rcRw > 0.0) (dcDw - dcRw - dwRc + rcRw) / rcRw else Double.NaN
    val xiErr = if (nRcRwSm[i] > 0) (1.0 + xi) / sqrt(nRcRwSm[i].toDouble()) else Double.NaN
    XiBinCW(rCentersSmall[i], xi, xiErr, nRcRwSm[i])
}

println("%-8s  %-12s  %-10s".format("r [km]", "ξ_CW", "±σ"))
xiCWSmall.forEach { b ->
    println("%-8.1f  %-12.4f  %-10.4f".format(b.rCenter, b.xi, b.xiErr))
}


D_C D_W (small) を計算中...
D_C R_W (small) を計算中...
D_W R_C (small) を計算中...
R_C R_W (small) を計算中...
完了
r [km]    ξ_CW          ±σ        
1.0       303.6136      56.5653   
2.0       480.9660      73.4991   
3.0       307.6863      32.3591   
4.0       376.0098      38.8856   
5.0       256.4460      20.4168   
6.0       294.3137      23.4199   
7.0       255.7729      18.1566   
8.0       248.2337      17.0373   
9.0       221.3772      14.0644   
10.0      225.8151      13.9331   
11.0      210.0554      12.1650   
12.0      183.6841      9.8020    
13.0      172.8123      9.0361    
14.0      169.0115      8.4065    
15.0      171.3126      8.7031    
16.0      158.8317      7.5013    
17.0      150.7459      6.9262    
18.0      149.7771      6.7497    
19.0      135.1518      5.9478    
20.0      131.5630      5.6321    
21.0      129.8981      5.5714    
22.0      123.1176      5.2309    
23.0      115.7662      4.7008    
24.0      101.0486      3.9048    
25.0      103.9216      4.

In [24]:
// --- ξ_CW 小スケール (線形ビン) を CSV に出力 ---
// 出力: output/xi_cw_small.csv
run {
    val file = java.io.File("./output/xi_cw_small.csv")
    file.bufferedWriter().use { w ->
        w.appendLine("r_km,xi_cw,xi_cw_err,n_rr")
        xiCWSmall.forEach { b ->
            w.appendLine("${b.rCenter},${b.xi},${b.xiErr},${b.nRR}")
        }
    }
    println("保存: ${file.path}  (${xiCWSmall.size} 行)")
}


保存: ./output/xi_cw_small.csv  (30 行)


In [25]:
// --- 小スケール ξ_CW 可視化 ---
// anchor clustering (r < 1 km, ξ_CW >> 0) と suburban 競合圏 (符号変化) を確認
val smPlotData = mapOf(
    "r"      to xiCWSmall.map { it.rCenter },
    "xi"     to xiCWSmall.map { it.xi },
    "xiLow"  to xiCWSmall.map { it.xi - it.xiErr },
    "xiHigh" to xiCWSmall.map { it.xi + it.xiErr }
)

letsPlot(smPlotData) +
    geomRibbon(alpha = 0.15, fill = "#E87040") {
        x = "r"; ymin = "xiLow"; ymax = "xiHigh"
    } +
    geomLineRange(size = 0.5, alpha = 0.7, color = "#E87040") {
        x = "r"; ymin = "xiLow"; ymax = "xiHigh"
    } +
    geomLine(size = 1.2, color = "#E87040") { x = "r"; y = "xi" } +
    geomPoint(size = 2.5, color = "#E87040") { x = "r"; y = "xi" } +
    geomHLine(yintercept = 0.0, linetype = "dashed", color = "#808080") +
    scaleXContinuous(name = "分離距離 r [km]") +
    scaleYContinuous(name = "ξ_CW(r)") +
    ggtitle(
        "小スケール交差相関 ξ_CW(r)：Coles–Woolworths 共存 vs 競合",
        "ξ_CW > 0 → anchor clustering（同一モール共存）、ξ_CW < 0 → suburban 競合排除"
    ) +
    theme(plotBackground = elementRect(fill = "#ffffff"))


<path d="M24.27537198334525 109.92880161027068 L24.27537198334525 109.92880161027068 L41.017007833928176 13.727272727272691 L57.75864368451111 119.89799304626203 L74.50027953509402 82.83571652630445 L91.24191538567695 151.18310032175586 L107.98355123625988 130.94577607053395 L124.72518708684281 152.6355211019022 L141.46682293742575 156.92283354846646 L158.20845878800867 171.69295498027532 L174.9500946385916 169.5604866209594 L191.69173048917452 178.239433047683 L208.43336633975747 192.46734036921566 L225.1750021903404 198.22984758222174 L241.91663804092332 200.42351331612971 L258.6582738915062 199.13728207186736 L275.39990974208916 205.91231543375332 L292.1415455926721 210.20078581718647 L308.883181443255 210.7679097829511 L325.62481729383796 218.40673561351406 L342.36645314442086 220.34008805691212 L359.1080889950038 221.1945039965696 L375.8497248455867 224.72046976450886 L392.59136069616966 228.62301722298437 L409.3329965467526 236.3046912081047 L426.0746323973355 234.83058399656645 L442.81626824791846 241.2242338239442 L459.55790409850135 243.033049105332 L476.2995399490843 245.77025139718643 L493.0411757996672 245.17913455946285 L509.78281165025015 251.08936938818354 L509.78281165025015 253.62618164186787 L493.0411757996672 248.17787833145337 L476.2995399490843 248.7785271946461 L459.55790409850135 246.2450734740897 L442.81626824791846 244.61126049110223 L426.0746323973355 238.80057481434682 L409.3329965467526 240.1716234905116 L392.59136069616966 233.27828527270353 L375.8497248455867 229.90070211534857 L359.1080889950038 226.7118960428957 L342.36645314442086 225.91756063940085 L325.62481729383796 224.2969042178127 L308.883181443255 217.45219653817855 L292.1415455926721 217.05987440565013 L275.39990974208916 213.34088416378586 L258.6582738915062 207.75602744964146 L241.91663804092332 208.74856183626693 L225.1750021903404 207.178334364565 L208.43336633975747 202.17434981520196 L191.69173048917452 190.28655607556652 L174.9500946385916 183.35858050173314 L158.20845878800867 185.62100469427025 L141.46682293742575 173.79496236236312 L124.72518708684281 170.61612347779482 L107.98355123625988 154.13867375514621 L91.24191538567695 171.40200275383717 L74.50027953509402 121.34441000691305 L57.75864368451111 151.94346354677648 L41.017007833928176 86.51399996991458 L24.27537198334525 165.945861397945 Z" fill="rgb(232,112,64)" stroke-width="1.0" fill-opacity="0.14901960784313725">
 
 
 
 <path d="M24.27537198334525 109.92880161027068 L24.27537198334525 109.92880161027068 L41.017007833928176 13.727272727272691 L57.75864368451111 119.89799304626203 L74.50027953509402 82.83571652630445 L91.24191538567695 151.18310032175586 L107.98355123625988 130.94577607053395 L124.72518708684281 152.6355211019022 L141.46682293742575 156.92283354846646 L158.20845878800867 171.69295498027532 L174.9500946385916 169.5604866209594 L191.69173048917452 178.239433047683 L208.43336633975747 192.46734036921566 L225.1750021903404 198.22984758222174 L241.91663804092332 200.42351331612971 L258.6582738915062 199.13728207186736 L275.39990974208916 205.91231543375332 L292.1415455926721 210.20078581718647 L308.883181443255 210.7679097829511 L325.62481729383796 218.40673561351406 L342.36645314442086 220.34008805691212 L359.1080889950038 221.1945039965696 L375.8497248455867 224.72046976450886 L392.59136069616966 228.62301722298437 L409.3329965467526 236.3046912081047 L426.0746323973355 234.83058399656645 L442.81626824791846 241.2242338239442 L459.55790409850135 243.033049105332 L476.2995399490843 245.77025139718643 L493.0411757996672 245.17913455946285 L509.78281165025015 251.08936938818354 " fill="none" stroke-width="1.6500000000000001" stroke="rgb(71,71,71)" stroke-opacity="1.0">
 
 
 
 <path d="M24.27537198334525 165.945861397945 L24.27537198334525 165.945861397945 L41.017007833928176 86.51399996991458 L57.75864368451111 151.94346354677648 L74.50027953509402 121.34441000691305 L91.24191538567695 171.40200275383717 L107.98355123625988 154.13867375514621 L124.72518708684

## Woolworths 自己相関 $\xi_{WW}(r)$：$\xi_{CC}$ vs $\xi_{CW}$ vs $\xi_{WW}$ の三者比較

**目的**: $\xi_{CW} > \xi_{CC}$（$r > 71$ km）の原因を切り分ける。

| 仮説 | $\xi_{WW}$ の予測 |
|---|---|
| Woolworths の店舗数優位（1039 vs 685）による密度効果 | $\xi_{WW} \approx \xi_{CC}$（同種クラスタリングは同程度）|
| Woolworths が空間的により均質に分布 | $\xi_{WW} < \xi_{CC}$（同種でも薄くクラスタリング）|
| 純粋な spatial co-location 効果 | $\xi_{WW} \approx \xi_{CW} \approx \xi_{CC}$（3者が一致）|

$\xi_{WW}$ は既存の `pairCounts` 関数と `randomMasked`（Coles と同じランダムカタログ）を流用して計算する。
ただし `woolworthsPoints` は `PointW` 型のため、`Point` 型への変換が必要。

In [26]:
// ⚠️ 前提: Cell 11–12 (isOnContinent, randomMasked) と Cell 19 (woolworthsPoints) が実行済みであること

// --- PointW → Point 変換 ---
val woolworthsAsPoints: List<Point> = woolworthsPoints.map { Point(it.lat, it.lon) }

// --- Woolworths 用大陸マスク済みランダムカタログ (Point 型, seed=456) ---
// randomMasked (seed=42) は Coles の DR/RR 計算に使用済みのため独立シードを使う
val rngWW = kotlin.random.Random(456)
val randomWW: List<Point> = generateSequence {
    val lat = rngWW.nextDouble(-44.0, -10.0)
    val lon = rngWW.nextDouble(113.0, 154.0)
    Point(lat, lon)
}.filter { isOnContinent(it.lat, it.lon) }
 .take(10 * nW)
 .toList()

val nRWW = randomWW.size
println("Woolworths 自己相関用ランダムカタログ: $nRWW 点")
println("Woolworths 店舗数: $nW")


Woolworths 自己相関用ランダムカタログ: 10390 点
Woolworths 店舗数: 1039


In [27]:
// --- Woolworths ペアカウント ---
println("DD_WW を計算中...")
val nDDww = pairCounts(woolworthsAsPoints, null, bins)

println("DR_WW を計算中...")
val nDRww = pairCounts(woolworthsAsPoints, randomWW, bins)

println("RR_WW を計算中...")
val nRRww = pairCounts(randomWW, null, bins)

println("完了")

// --- Woolworths Landy-Szalay 推定量 ---
val normDDww = nW.toLong() * (nW - 1) / 2
val normDRww = nW.toLong() * nRWW
val normRRww = nRWW.toLong() * (nRWW - 1) / 2

val xiWW: List<XiBin> = (0 until nBins).map { i ->
    val dd = nDDww[i].toDouble() / normDDww
    val dr = nDRww[i].toDouble() / normDRww
    val rr = nRRww[i].toDouble() / normRRww
    val xi = if (rr > 0.0) (dd - 2.0 * dr + rr) / rr else Double.NaN
    val xiErr = if (nRRww[i] > 0) (1.0 + xi) / sqrt(nRRww[i].toDouble()) else Double.NaN
    XiBin(rCenters[i], xi, xiErr, nRRww[i])
}

// --- 三者比較テーブル ---
println("%-10s  %-10s  %-10s  %-10s  %-10s  %-10s"
    .format("r [km]", "ξ_CC", "ξ_CW", "ξ_WW", "CC-CW", "CC-WW"))
println("-".repeat(65))
(0 until nBins).forEach { i ->
    val cc  = xiMasked[i].xi
    val cw  = xiCW[i].xi
    val ww  = xiWW[i].xi
    println("%-10.1f  %-10.4f  %-10.4f  %-10.4f  %-10.4f  %-10.4f"
        .format(rCenters[i], cc, cw, ww, cc - cw, cc - ww))
}


DD_WW を計算中...
DR_WW を計算中...
RR_WW を計算中...
完了
r [km]      ξ_CC        ξ_CW        ξ_WW        CC-CW       CC-WW     
-----------------------------------------------------------------
8.2         229.5966    230.1173    219.3424    -0.5206     10.2543   
9.5         263.1330    232.2466    205.0761    30.8864     58.0569   
11.0        217.6067    207.1814    194.7699    10.4254     22.8368   
12.7        185.7089    174.8560    182.2459    10.8529     3.4631    
14.8        184.4402    169.0676    155.1007    15.3727     29.3395   
17.1        159.9554    150.4283    153.1026    9.5271      6.8528    
19.8        144.5967    134.3201    129.9405    10.2766     14.6562   
22.9        118.7438    112.6759    110.7357    6.0679      8.0081    
26.5        98.4448     90.9942     86.6938     7.4507      11.7511   
30.7        78.6296     71.8487     67.7424     6.7810      10.8872   
35.5        55.4220     53.2014     53.6392     2.2206      1.7828    
41.1        43.0323     38.5237     3

In [28]:
// --- ξ_CC / ξ_CW / ξ_WW を CSV に出力 ---
// 出力: output/xi_all_wide.csv
run {
    val file = java.io.File("./output/xi_all_wide.csv")
    file.bufferedWriter().use { w ->
        w.appendLine("# nD=$nD, nW=$nW, nRMasked=$nRMasked, nRWW=$nRWW, nBins=$nBins")
        w.appendLine("r_km,xi_cc,xi_cc_err,xi_cw,xi_cw_err,xi_ww,xi_ww_err")
        (0 until nBins).forEach { i ->
            val cc = xiMasked[i]; val cw = xiCW[i]; val ww = xiWW[i]
            w.appendLine("${cc.rCenter},${cc.xi},${cc.xiErr},${cw.xi},${cw.xiErr},${ww.xi},${ww.xiErr}")
        }
    }
    println("保存: ${file.path}  (${nBins} 行)")
}


保存: ./output/xi_all_wide.csv  (38 行)


In [29]:
// --- 三者比較プロット: ξ_CC vs ξ_CW vs ξ_WW ---
//
// 読み方:
//   ξ_CC ≈ ξ_WW > ξ_CW  → 同種クラスタリングが強い（micro-partitioning）
//   ξ_CC > ξ_WW ≈ ξ_CW  → Coles が特異的にクラスター（Woolworths は均質）
//   ξ_CC ≈ ξ_CW ≈ ξ_WW  → 3者が人口密度場を等しく追随

val r3     = rCenters + rCenters + rCenters
val xi3    = xiMasked.map { it.xi }   + xiCW.map { it.xi }    + xiWW.map { it.xi }
val xiL3   = xiMasked.map { it.xi - it.xiErr } + xiCW.map { it.xi - it.xiErr } + xiWW.map { it.xi - it.xiErr }
val xiH3   = xiMasked.map { it.xi + it.xiErr } + xiCW.map { it.xi + it.xiErr } + xiWW.map { it.xi + it.xiErr }
val lbl3   = List(nBins) { "ξ_CC  Coles" } + List(nBins) { "ξ_CW  Coles×Woolworths" } + List(nBins) { "ξ_WW  Woolworths" }

val triData = mapOf("r" to r3, "xi" to xi3, "xiLow" to xiL3, "xiHigh" to xiH3, "label" to lbl3)

letsPlot(triData) +
    geomRibbon(alpha = 0.10) {
        x = "r"; ymin = "xiLow"; ymax = "xiHigh"; fill = "label"
    } +
    geomLine(size = 1.3) { x = "r"; y = "xi"; color = "label" } +
    geomPoint(size = 1.8) { x = "r"; y = "xi"; color = "label" } +
    geomHLine(yintercept = 0.0, linetype = "dashed", color = "#808080") +
    geomVLine(xintercept = 71.3, linetype = "dotted", color = "#666666") +
    geomVLine(xintercept = 666.0, linetype = "dotted", color = "#CC4444") +
    scaleXLog10(name = "分離距離 r [km]") +
    scaleYContinuous(name = "ξ(r)") +
    scaleColorManual(values = mapOf(
        "ξ_CC  Coles"             to "#4682B4",
        "ξ_CW  Coles×Woolworths" to "#E87040",
        "ξ_WW  Woolworths"        to "#3DAA5C"
    )) +
    scaleFillManual(values = mapOf(
        "ξ_CC  Coles"             to "#4682B4",
        "ξ_CW  Coles×Woolworths" to "#E87040",
        "ξ_WW  Woolworths"        to "#3DAA5C"
    )) +
    ggtitle(
        "ξ_CC vs ξ_CW vs ξ_WW：duopoly 三者比較",
        "灰点線=r=71.3 km (Duopoly Debye Length)、赤点線=r=666 km (Retail BAO)"
    ) +
    theme(plotBackground = elementRect(fill = "#ffffff"))


<path d="M0.0 47.412411663036835 L0.0 47.412411663036835 L8.132385715815488 13.727272727272691 L16.264771431631004 63.37829019727047 L24.397157147446478 97.7825612810482 L32.52954286326198 99.96205531932051 L40.66192857907748 125.79721212507164 L48.794314294892956 141.9201133057716 L56.92670001070846 168.5206899996041 L65.05908572652393 189.22533822702712 L73.19147144233943 209.252825027562 L81.32385715815494 232.5683213885778 L89.45624287397044 244.97342880973613 L97.58862858978591 254.52207276134482 L105.72101430560141 263.64691582366663 L113.85340002141692 270.87202462114067 L121.98578573723242 276.62565159015116 L130.1181714530479 280.05701218735305 L138.2505571688634 282.4785646769555 L146.38294288467884 282.51778844795643 L154.51532860049437 284.1414275887383 L162.64771431630984 284.94947570159405 L170.78010003212538 286.08935059625196 L178.91248574794085 286.5570668403062 L187.04487146375638 285.88680044672327 L195.17725717957185 286.98981825545764 L203.30964289538733 287.26274539611217 L211.44202861120286 287.1300897473248 L219.57441432701827 287.07806949665485 L227.7068000428338 287.2812912813047 L235.83918575864934 286.97004038885916 L243.9715714744648 284.52989838201046 L252.10395719028023 286.23869748466154 L260.23634290609584 287.45762829858586 L268.36872862191126 287.72302738641804 L276.5011143377268 287.5001084569913 L284.6335000535422 287.21110586228997 L292.76588576935774 287.8751580708579 L300.89827148517327 288.20684265732297 L300.89827148517327 288.2074351627665 L292.76588576935774 287.87619480292324 L284.6335000535422 287.2130926190872 L276.5011143377268 287.5018135508503 L268.36872862191126 287.7245331609518 L260.23634290609584 287.4597541711623 L252.10395719028023 286.2434391590523 L243.9715714744648 284.538968732432 L235.83918575864934 286.9742420270612 L227.7068000428338 287.2851770200348 L219.57441432701827 287.0831293111558 L211.44202861120286 287.1356636465657 L203.30964289538733 287.26855465121804 L195.17725717957185 286.99774680113086 L187.04487146375638 285.90182006950954 L178.91248574794085 286.570126697142 L170.78010003212538 286.10764296977914 L162.64771431630984 284.9797536198612 L154.51532860049437 284.18377477515463 L146.38294288467884 282.58414376135465 L138.2505571688634 282.5552511204926 L130.1181714530479 280.17991794878407 L121.98578573723242 276.8247046993468 L113.85340002141692 271.21012271612767 L105.72101430560141 264.1958727427019 L97.58862858978591 255.38396638684304 L89.45624287397044 246.24822230550274 L81.32385715815494 234.42072797516937 L73.19147144233943 212.34493940534279 L65.05908572652393 193.61644139522627 L56.92670001070846 174.61883844816202 L48.794314294892956 150.65436890531302 L40.66192857907748 136.73756672251199 L32.52954286326198 114.68350502695884 L24.397157147446478 114.38161523814512 L16.264771431631004 86.39774153483904 L8.132385715815488 47.00511130188178 L0.0 78.91288018633514 Z" fill="rgb(70,130,180)" stroke-width="1.0" fill-opacity="0.10196078431372549">
 
 
 
 <path d="M0.0 49.07337086745997 L0.0 49.07337086745997 L8.132385715815488 48.254901189131914 L16.264771431631004 75.92348955401289 L24.397157147446478 110.20005261045947 L32.52954286326198 116.73699625526493 L40.66192857907748 136.34801485120263 L48.794314294892956 152.9855754014475 L56.92670001070846 175.0697386844198 L65.05908572652393 197.02020797471698 L73.19147144233943 216.26778849228117 L81.32385715815494 234.92472946676057 L89.45624287397044 249.55326588853222 L97.58862858978591 258.60185515352254 L105.72101430560141 266.421222595651 L113.85340002141692 272.0002830748977 L121.98578573723242 276.2466900888343 L130.1181714530479 279.8056339524101 L138.2505571688634 282.3164767819775 L146.38294288467884 282.677193514738 L154.51532860049437 283.86739389987605 L162.64771431630984 284.8681662233034 L170.78010003212538 286.01610033617385 L178.91248574794085 286.4068847337557 L187.04487146375638 285.83975230130056 L195.17725717957185 286.8386865666818 L203.30964289538733 287.1094723749877 L211.442

## Bias モデル検証：$\xi_{CW}^{\rm pred} = \sqrt{\xi_{CC} \cdot \xi_{WW}}$ vs 実測 $\xi_{CW}$

線形 bias モデル（$\delta_C = b_C \delta_m$、$\delta_W = b_W \delta_m$）が成立するなら：

$$\xi_{CW}(r) = b_C b_W \xi_{mm}(r) = \sqrt{\xi_{CC}(r) \cdot \xi_{WW}(r)}$$

残差 $\Delta\xi_{CW}(r) \equiv \xi_{CW}(r) - \sqrt{\xi_{CC}(r) \cdot \xi_{WW}(r)}$ が
- $\approx 0$：bias モデルが成立（両社は独立に同一密度場を追随）
- $> 0$：モデル予測を超える共存（attractive interaction）
- $< 0$：モデル予測を下回る共存（repulsive interaction）


In [30]:
// --- bias モデル計算 ---
data class BiasBin(
    val rCenter: Double,
    val xiCC: Double, val xiCW: Double, val xiWW: Double,
    val xiPred: Double,   // sqrt(xi_CC * xi_WW)
    val residual: Double, // xi_CW - xi_pred
    val bwBc: Double      // b_W / b_C = sqrt(xi_WW / xi_CC)
)

val biasResult: List<BiasBin> = (0 until nBins).map { i ->
    val cc = xiMasked[i].xi
    val cw = xiCW[i].xi
    val ww = xiWW[i].xi
    val pred   = if (cc > 0 && ww > 0) sqrt(cc * ww) else Double.NaN
    val resid  = if (pred.isNaN()) Double.NaN else cw - pred
    val ratio  = if (cc > 0 && ww > 0) sqrt(ww / cc) else Double.NaN
    BiasBin(rCenters[i], cc, cw, ww, pred, resid, ratio)
}

// --- テーブル出力 ---
println("%-8s  %-8s  %-8s  %-8s  %-8s  %-10s  %-8s"
    .format("r [km]", "ξ_CC", "ξ_CW", "ξ_WW", "pred", "Δξ_CW", "b_W/b_C"))
println("-".repeat(70))
biasResult.forEach { b ->
    println("%-8.1f  %-8.4f  %-8.4f  %-8.4f  %-8.4f  %-10.4f  %-8.4f"
        .format(b.rCenter, b.xiCC, b.xiCW, b.xiWW,
                if (b.xiPred.isNaN()) Double.NaN else b.xiPred,
                if (b.residual.isNaN()) Double.NaN else b.residual,
                if (b.bwBc.isNaN()) Double.NaN else b.bwBc))
}


r [km]    ξ_CC      ξ_CW      ξ_WW      pred      Δξ_CW       b_W/b_C 
----------------------------------------------------------------------
8.2       229.5966  230.1173  219.3424  224.4109  5.7063      0.9774  
9.5       263.1330  232.2466  205.0761  232.2978  -0.0512     0.8828  
11.0      217.6067  207.1814  194.7699  205.8719  1.3095      0.9461  
12.7      185.7089  174.8560  182.2459  183.9692  -9.1132     0.9906  
14.8      184.4402  169.0676  155.1007  169.1355  -0.0679     0.9170  
17.1      159.9554  150.4283  153.1026  156.4915  -6.0632     0.9783  
19.8      144.5967  134.3201  129.9405  137.0729  -2.7528     0.9480  
22.9      118.7438  112.6759  110.7357  114.6699  -1.9939     0.9657  
26.5      98.4448   90.9942   86.6938   92.3826   -1.3885     0.9384  
30.7      78.6296   71.8487   67.7424   72.9833   -1.1346     0.9282  
35.5      55.4220   53.2014   53.6392   54.5233   -1.3219     0.9838  
41.1      43.0323   38.5237   36.2356   39.4880   -0.9643     0.9176  
47.6  

In [31]:
// --- プロット1: ξ_CW 実測 vs 幾何平均予測 ---
val validBias = biasResult.filter { !it.xiPred.isNaN() && it.rCenter < 900 }

val bpData = mapOf(
    "r"     to (validBias.map { it.rCenter } + validBias.map { it.rCenter }),
    "xi"    to (validBias.map { it.xiCW }    + validBias.map { it.xiPred }),
    "label" to (List(validBias.size) { "実測 ξ_CW" } + List(validBias.size) { "予測 √(ξ_CC·ξ_WW)" })
)

letsPlot(bpData) +
    geomLine(size = 1.3) { x = "r"; y = "xi"; color = "label" } +
    geomPoint(size = 2.0) { x = "r"; y = "xi"; color = "label" } +
    scaleXLog10(name = "分離距離 r [km]") +
    scaleYLog10(name = "ξ(r)  [log scale]") +
    scaleColorManual(values = mapOf(
        "実測 ξ_CW"          to "#E87040",
        "予測 √(ξ_CC·ξ_WW)" to "#888888"
    )) +
    ggtitle(
        "bias モデル検証: ξ_CW vs √(ξ_CC · ξ_WW)",
        "灰=幾何平均予測。両者が一致するほど linear bias モデルが成立"
    ) +
    theme(plotBackground = elementRect(fill = "#ffffff"))


<path d="M0.0 2.630212491328763 L0.0 2.630212491328763 L10.894640869538392 2.2136661832126663 L21.789281739076813 7.378583622943324 L32.68392260861518 15.05016576702269 L43.5785634781536 16.57264644231205 L54.47320434769202 21.855449899669907 L65.36784521723038 26.977691031322593 L76.2624860867688 34.924149190788256 L87.15712695630717 44.589658438637656 L98.05176782584559 55.27327687048751 L108.94640869538398 68.86241103348053 L119.8410495649224 83.46147425506209 L130.73569043446082 95.64248681383927 L141.63033130399918 109.7402290134228 L152.5249721735376 123.44342282827037 L163.41961304307597 137.67091294704989 L174.3142539126144 154.47712983500614 L185.2088947821528 171.77943352993424 L196.10353565169117 174.8861796461914 L206.9981765212296 187.11409363576297 L217.89281739076802 200.8113850336289 L228.78745826030638 224.39281190609717 L239.68209912984474 236.37900548422013 L250.57673999938322 219.81624370652847 L261.4713808689216 254.85207318886341 L272.36602173846 272.0912153253662 L283.26066260799837 269.60706448019795 L294.15530347753673 259.4657004416535 L305.04994434707515 285.86590861760834 L315.94458521661363 259.75611332425024 L326.839226086152 197.835483868555 L337.7338669556903 229.94988143471318 L348.6285078252288 302.0 " fill="none" stroke-width="2.8600000000000003" stroke="rgb(232,112,64)" stroke-opacity="1.0">
 
 
 
 <path d="M0.0 3.765819557079652 L0.0 3.765819557079652 L10.894640869538392 2.203687570773809 L21.789281739076813 7.665327924410519 L32.68392260861518 12.752489551544755 L43.5785634781536 16.5544850463383 L54.47320434769202 20.06839241263077 L65.36784521723038 26.060204072370567 L76.2624860867688 34.130840364423364 L87.15712695630717 43.904781436033204 L98.05176782584559 54.56468845982366 L108.94640869538398 67.75239446545629 L119.8410495649224 82.34339597292563 L130.73569043446082 95.50290217849331 L141.63033130399918 109.5508633307499 L152.5249721735376 122.52583673155495 L163.41961304307597 138.2751785380607 L174.3142539126144 154.26722378919294 L185.2088947821528 171.1244040622991 L196.10353565169117 174.2071244925069 L206.9981765212296 187.40307583330227 L217.89281739076802 199.71926583202588 L228.78745826030638 222.86698073998605 L239.68209912984474 235.23465567896892 L250.57673999938322 219.00064783483708 L261.4713808689216 256.76700575215614 L272.36602173846 272.5343278177659 L283.26066260799837 267.0203566631754 L294.15530347753673 261.35338498261456 L305.04994434707515 283.23636987602447 L315.94458521661363 257.52848991654486 L326.839226086152 197.78292800019125 L337.7338669556903 230.2345683092419 L348.6285078252288 299.98563751419897 " fill="none" stroke-width="2.8600000000000003" stroke="rgb(136,136,136)" stroke-opacity="1.0">
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 
 
 32 
 
 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 316 
 
 
 
 
 
 
 
 
 
 
 1 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 bias モデル検証: ξ_CW vs √(ξ_CC · ξ_WW) 
 
 
 
 
 灰=幾何平均予測。両者が一致するほど linear bias モデルが成立 
 
 
 
 
 ξ(r) [log scale] 
 
 
 
 
 分離距離 r [km] 
 
 
 
 
 
 
 
 
 label 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 実測 ξ_CW 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 予測 √(ξ_CC·ξ_WW)

In [32]:
// --- プロット2: 残差 Δξ_CW(r) ---
val resData = mapOf(
    "r"      to validBias.map { it.rCenter },
    "resid"  to validBias.map { it.residual },
    "bw_bc"  to validBias.map { it.bwBc }
)

val pResid = letsPlot(resData) +
    geomHLine(yintercept = 0.0, linetype = "dashed", color = "#808080") +
    geomVLine(xintercept = 71.3,  linetype = "dotted", color = "#666666") +
    geomVLine(xintercept = 666.0, linetype = "dotted", color = "#CC4444") +
    geomLine(size = 1.1, color = "#4682B4") { x = "r"; y = "resid" } +
    geomPoint(size = 2.0, color = "#4682B4") { x = "r"; y = "resid" } +
    scaleXLog10(name = "分離距離 r [km]") +
    scaleYContinuous(name = "Δξ_CW = ξ_CW − √(ξ_CC·ξ_WW)") +
    ggtitle(
        "残差 Δξ_CW(r)：bias モデルからの逸脱",
        "Δ>0: 予測超過（attractive）、Δ<0: 予測未達（repulsive）。灰点線=71.3 km、赤点線=666 km"
    ) +
    theme(plotBackground = elementRect(fill = "#ffffff"))

val pBias = letsPlot(resData) +
    geomHLine(yintercept = 1.0, linetype = "dashed", color = "#808080") +
    geomVLine(xintercept = 71.3,  linetype = "dotted", color = "#666666") +
    geomVLine(xintercept = 666.0, linetype = "dotted", color = "#CC4444") +
    geomLine(size = 1.1, color = "#3DAA5C") { x = "r"; y = "bw_bc" } +
    geomPoint(size = 2.0, color = "#3DAA5C") { x = "r"; y = "bw_bc" } +
    scaleXLog10(name = "分離距離 r [km]") +
    scaleYContinuous(name = "b_W / b_C = √(ξ_WW / ξ_CC)") +
    ggtitle(
        "スケール依存バイアス比 b_W(r) / b_C(r)",
        ">1: Woolworths が高バイアス（都市集中）、<1: Coles が高バイアス。赤点線=Retail BAO"
    ) +
    theme(plotBackground = elementRect(fill = "#ffffff"))

pResid


<path d="M0.0 13.727272727272748 L0.0 13.727272727272748 L16.76585441478241 120.39198980214651 L33.531708829564906 95.18364919712853 L50.29756324434729 288.27272727272725 L67.06341765912973 120.7005978910471 L83.82927207391222 231.7681045830945 L100.5951264886946 170.44078061181122 L117.3609809034771 156.3819366782893 L134.12683531825948 145.16546659144598 L150.89268973304192 140.46199380073492 L167.65854414782441 143.9328647390814 L184.42439856260685 137.3066623194295 L201.19025297738924 121.12779443334088 L217.95610739217173 121.11743649358942 L234.7219618069541 125.48542681382754 L251.48781622173655 116.58569720367456 L268.253670636519 120.13310597457289 L285.01952505130146 120.91969422940633 L301.7853794660839 120.872591696596 L318.55123388086633 118.9830947232041 L335.3170882956488 120.74494926635452 L352.0829427104312 120.52802762795596 L368.84879712521354 120.06446108275635 L385.6146515399962 120.07951012042086 L402.3805059547785 118.77380147374345 L419.14636036956097 119.3351203719706 L435.9122147843434 120.12772540594135 L452.67806919912573 118.84707382712833 L469.4439236139083 119.92896542512501 L486.20977802869083 120.17327734060774 L502.97563244347316 119.50871822294619 L519.7414868582555 119.2669945935883 L536.507341273038 119.70157894429448 " fill="none" stroke-width="2.4200000000000004" stroke="rgb(70,130,180)" stroke-opacity="1.0">
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 
 
 16 
 
 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 
 
 63 
 
 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 158 
 
 
 
 
 
 
 
 
 251 
 
 
 
 
 
 
 
 
 398 
 
 
 
 
 
 
 
 
 631 
 
 
 
 
 
 
 
 
 
 
 -8 
 
 
 
 
 
 
 -6 
 
 
 
 
 
 
 -4 
 
 
 
 
 
 
 -2 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 
 
 残差 Δξ_CW(r)：bias モデルからの逸脱 
 
 
 
 
 Δ>0: 予測超過（attractive）、Δ<0: 予測未達（repulsive）。灰点線=71.3 km、赤点線=666 km 
 
 
 
 
 Δξ_CW = ξ_CW − √(ξ_CC·ξ_WW) 
 
 
 
 
 分離距離 r [km]

In [33]:
// b_W/b_C プロット
pBias


<path d="M0.0 231.59385577826345 L0.0 231.59385577826345 L16.608925793411544 287.6080737523898 L33.217851586823144 250.1514032764071 L49.82677738023466 223.76692045025197 L66.43570317364623 267.3542654647606 L83.0446289670578 231.042840888024 L99.65355476046932 249.03027793939964 L116.26248055388095 238.53500496691413 L132.87140634729246 254.68282511435132 L149.48033214070404 260.7402490128021 L166.0892579341156 227.82144469501532 L182.69818372752718 266.99014209117104 L199.3071095209387 288.27272727272714 L215.91603531435027 283.30387713621224 L232.52496110776184 245.62350472000662 L249.1338869011734 205.10484145039118 L265.7428126945849 195.01562648058393 L282.3517384879965 190.10085687729986 L298.96066428140807 227.1336005451809 L315.56959007481964 175.90870954766808 L332.1785158682312 185.16219281191968 L348.7874416616428 169.25938265029401 L365.39636745505425 121.94254471390502 L382.00529324846593 191.2796439486525 L398.6142190418774 119.90104446262399 L415.22314483528896 13.727272727272748 L431.83207062870054 123.46696234490116 L448.440996422112 98.15362772806498 L465.04992221552357 154.661960706434 L481.65884800893525 151.79030799881718 L498.2677738023468 246.6639289232926 L514.8766995957583 212.17512400726105 L531.48562538917 19.650223774702113 " fill="none" stroke-width="2.4200000000000004" stroke="rgb(61,170,92)" stroke-opacity="1.0">
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 
 
 16 
 
 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 
 
 63 
 
 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 158 
 
 
 
 
 
 
 
 
 251 
 
 
 
 
 
 
 
 
 398 
 
 
 
 
 
 
 
 
 631 
 
 
 
 
 
 
 
 
 
 
 0.9 
 
 
 
 
 
 
 1 
 
 
 
 
 
 
 1.1 
 
 
 
 
 
 
 1.2 
 
 
 
 
 
 
 1.3 
 
 
 
 
 
 
 
 
 スケール依存バイアス比 b_W(r) / b_C(r) 
 
 
 
 
 >1: Woolworths が高バイアス（都市集中）、<1: Coles が高バイアス。赤点線=Retail BAO 
 
 
 
 
 b_W / b_C = √(ξ_WW / ξ_CC) 
 
 
 
 
 分離距離 r [km]

In [34]:
// --- bias 解析結果を CSV に保存 ---
// 出力: output/xi_bias_analysis.csv
run {
    val file = java.io.File("./output/xi_bias_analysis.csv")
    file.bufferedWriter().use { w ->
        w.appendLine("# Geometric-mean bias model: xi_cw_pred = sqrt(xi_cc * xi_ww)")
        w.appendLine("# residual = xi_cw - xi_cw_pred, bw_bc = sqrt(xi_ww/xi_cc)")
        w.appendLine("r_km,xi_cc,xi_cw,xi_ww,xi_cw_pred,residual,bw_bc")
        biasResult.forEach { b ->
            w.appendLine("${b.rCenter},${b.xiCC},${b.xiCW},${b.xiWW},${b.xiPred},${b.residual},${b.bwBc}")
        }
    }
    println("保存: ${file.path}")
}


保存: ./output/xi_bias_analysis.csv


## 空間ジャックナイフ誤差評価

Poisson 誤差 $\sigma_\xi^{\rm Poisson} = (1+\xi)/\sqrt{n_{RR}}$ は**ショットノイズの下限**に過ぎず、
以下の誤差源を無視している：

- **宇宙分散（Cosmic Variance）**: 調査領域がひとつしかない場合、サンプリングゆらぎを見積もれない
- **システマティクス**: 人口密度勾配・沿岸効果などの大スケール相関

空間ジャックナイフ推定量（Delete-$k$-out）：

$$\sigma_{\rm JK}^2(r) = \frac{K-1}{K} \sum_{k=1}^{K} \left[ \hat{\xi}_k(r) - \bar{\xi}_{\rm JK}(r) \right]^2$$

- $\hat{\xi}_k(r)$：パッチ $k$ を除いた $N - n_k$ 店舗で計算した LS 推定量
- $\bar{\xi}_{\rm JK}(r)$：$K$ 個の leave-one-out 推定量の平均
- $K = 8$：経度 $113$–$154°$ を等分割（$\Delta {\rm lon} \approx 5.1°$/パッチ）

ジャックナイフ $\sigma$ は大スケールで Poisson $\sigma$ を大幅に上回り、
Retail BAO シグナルの実際の有意性を評価する。

In [35]:
// --- 空間ジャックナイフ: パッチ割り当て ---
// K = 8 経度スライス (113–154° を等分割)
val K = 8
val lonBandEdges = DoubleArray(K + 1) { k -> 113.0 + k * (154.0 - 113.0) / K }

fun lonToPatch(lon: Double): Int {
    val idx = ((lon - 113.0) / (154.0 - 113.0) * K).toInt()
    return idx.coerceIn(0, K - 1)
}

val dataPatchIdx: List<Int> = dataPoints.map { lonToPatch(it.lon) }
val randPatchIdx: List<Int> = randomMasked.map { lonToPatch(it.lon) }

println("=== Coles 店舗分布 (経度パッチ) ===")
(0 until K).forEach { k ->
    val cnt = dataPatchIdx.count { it == k }
    val lo = lonBandEdges[k]; val hi = lonBandEdges[k + 1]
    println("  Patch $k [lon ${lo.toInt()}–${hi.toInt()}°]: $cnt 店舗")
}
println("合計: ${dataPatchIdx.size} 店舗")
println()
println("=== ランダムカタログ分布 ===")
(0 until K).forEach { k ->
    val cnt = randPatchIdx.count { it == k }
    println("  Patch $k: $cnt 点")
}

=== Coles 店舗分布 (経度パッチ) ===
  Patch 0 [lon 113–118°]: 85 店舗
  Patch 1 [lon 118–123°]: 3 店舗
  Patch 2 [lon 123–128°]: 0 店舗
  Patch 3 [lon 128–133°]: 7 店舗
  Patch 4 [lon 133–138°]: 22 店舗
  Patch 5 [lon 138–143°]: 29 店舗
  Patch 6 [lon 143–148°]: 209 店舗
  Patch 7 [lon 148–154°]: 330 店舗
合計: 685 店舗

=== ランダムカタログ分布 ===
  Patch 0: 528 点
  Patch 1: 715 点
  Patch 2: 886 点
  Patch 3: 903 点
  Patch 4: 1010 点
  Patch 5: 1197 点
  Patch 6: 1132 点
  Patch 7: 479 点


In [36]:
// --- ジャックナイフ leave-one-out ループ ---
// 各パッチ k を除いた (N - n_k) 点で DD, DR, RR を再計算し ξ_k(r) を求める
// ⚠️ 前提: Cell 11 (isOnContinent), Cell 12 (randomMasked), Cell 15 (xiMasked) が実行済み

println("ジャックナイフ計算開始 (K=$K)...")

val xiJK = Array(K) { DoubleArray(nBins) { Double.NaN } }   // [patch_k][bin]

for (k in 0 until K) {
    val dataK: List<Point> = dataPoints.filterIndexed { i, _ -> dataPatchIdx[i] != k }
    val randK: List<Point> = randomMasked.filterIndexed { i, _ -> randPatchIdx[i] != k }
    val nDk = dataK.size
    val nRk = randK.size

    if (nDk < 10) {
        println("  Patch $k: データ点不足でスキップ (nD=$nDk)")
        continue
    }

    val ddK = pairCounts(dataK, null, bins)
    val drK = pairCounts(dataK, randK, bins)
    val rrK = pairCounts(randK, null, bins)

    val normDDk = nDk.toLong() * (nDk - 1) / 2
    val normDRk = nDk.toLong() * nRk
    val normRRk = nRk.toLong() * (nRk - 1) / 2

    for (b in 0 until nBins) {
        val dd = ddK[b].toDouble() / normDDk
        val dr = drK[b].toDouble() / normDRk
        val rr = if (normRRk > 0L) rrK[b].toDouble() / normRRk else 0.0
        xiJK[k][b] = if (rr > 0.0) (dd - 2.0 * dr + rr) / rr else Double.NaN
    }
    println("  Patch $k 完了  (nD=$nDk, nR=$nRk)")
}
println("全パッチ完了")

// --- ジャックナイフ σ の計算 ---
// σ²_JK(r) = (K-1)/K × Σ_k [ ξ_k - ξ̄_JK ]²
val sigmaJK = DoubleArray(nBins) { b ->
    val vals = (0 until K).mapNotNull { k -> xiJK[k][b].takeIf { !it.isNaN() } }
    val n = vals.size
    if (n < 2) return@DoubleArray Double.NaN
    val mean = vals.average()
    sqrt((n - 1.0) / n * vals.sumOf { (it - mean) * (it - mean) })
}

// --- σ_JK vs σ_Poisson の比較表 ---
println()
println("%-8s  %-9s  %-9s  %-9s  %-8s".format("r [km]", "ξ_CC", "σ_Poisson", "σ_JK", "JK/Poisson"))
println("-".repeat(58))
(0 until nBins).forEach { b ->
    val xi = xiMasked[b].xi
    val sP = xiMasked[b].xiErr
    val sJ = sigmaJK[b]
    val ratio = if (!sP.isNaN() && !sJ.isNaN() && sP > 0) sJ / sP else Double.NaN
    println("%-8.1f  %-9.4f  %-9.4f  %-9.4f  %-8.2f".format(rCenters[b], xi, sP, sJ, ratio))
}

ジャックナイフ計算開始 (K=8)...
  Patch 0 完了  (nD=600, nR=6322)
  Patch 1 完了  (nD=682, nR=6135)
  Patch 2 完了  (nD=685, nR=5964)
  Patch 3 完了  (nD=678, nR=5947)
  Patch 4 完了  (nD=663, nR=5840)
  Patch 5 完了  (nD=656, nR=5653)
  Patch 6 完了  (nD=476, nR=5718)
  Patch 7 完了  (nD=355, nR=6371)
全パッチ完了

r [km]    ξ_CC       σ_Poisson  σ_JK       JK/Poisson
----------------------------------------------------------
8.2       229.5966   16.1056    178.4382   11.08   
9.5       263.1330   17.0143    236.2692   13.89   
11.0      217.6067   11.7694    198.1863   16.84   
12.7      185.7089   8.4868     169.0126   19.91   
14.8      184.4402   7.5268     167.1057   22.20   
17.1      159.9554   5.5936     153.7579   27.49   
19.8      144.5967   4.4657     134.9732   30.22   
22.9      118.7438   3.1179     107.4507   34.46   
26.5      98.4448    2.2451     91.9859    40.97   
30.7      78.6296    1.5809     72.9552    46.15   
35.5      55.4220    0.9471     53.7299    56.73   
41.1      43.0323    0.6518   

In [37]:
// --- σ_JK vs σ_Poisson: 比較プロット ---
// 上段: ξ_CC のエラーバー比較 (Poisson vs Jackknife)
// 下段: JK/Poisson 比率

val rPlot   = rCenters.toList()
val xiPlot  = (0 until nBins).map { xiMasked[it].xi }
val sP_list = (0 until nBins).map { xiMasked[it].xiErr }
val sJ_list = sigmaJK.toList()
val ratio_list = (0 until nBins).map { b ->
    val sP = sP_list[b]; val sJ = sJ_list[b]
    if (!sP.isNaN() && !sJ.isNaN() && sP > 0) sJ / sP else Double.NaN
}

// ξ_CC エラーバー比較用 tidy data (2 系列)
val rLong  = rPlot + rPlot
val xiLong = xiPlot + xiPlot
val loLong = (0 until nBins).map { xiPlot[it] - sP_list[it] } +
             (0 until nBins).map { xiPlot[it] - sJ_list[it] }
val hiLong = (0 until nBins).map { xiPlot[it] + sP_list[it] } +
             (0 until nBins).map { xiPlot[it] + sJ_list[it] }
val lblLong = List(nBins) { "Poisson σ" } + List(nBins) { "Jackknife σ" }

val pXi = letsPlot(mapOf("r" to rLong, "xi" to xiLong, "lo" to loLong, "hi" to hiLong, "lbl" to lblLong)) +
    geomRibbon(alpha = 0.15) { x = "r"; ymin = "lo"; ymax = "hi"; fill = "lbl" } +
    geomLine(size = 1.2) { x = "r"; y = "xi" } +
    geomHLine(yintercept = 0.0, linetype = "dashed", color = "#888888") +
    geomVLine(xintercept = 666.0, linetype = "dotted", color = "#CC4444") +
    scaleXLog10(name = "分離距離 r [km]") +
    scaleYContinuous(name = "ξ_CC(r)") +
    scaleFillManual(values = mapOf("Poisson σ" to "#4682B4", "Jackknife σ" to "#E87040")) +
    ggtitle("ξ_CC: Poisson σ vs Jackknife σ",
            "赤縦線: Retail BAO (r = 666 km)") +
    ggsize(800, 400)

val pRatio = letsPlot(mapOf("r" to rPlot, "ratio" to ratio_list)) +
    geomLine(size = 1.2, color = "#8B1A1A") { x = "r"; y = "ratio" } +
    geomPoint(size = 2.0, color = "#8B1A1A") { x = "r"; y = "ratio" } +
    geomHLine(yintercept = 1.0, linetype = "dashed", color = "#888888") +
    geomVLine(xintercept = 666.0, linetype = "dotted", color = "#CC4444") +
    scaleXLog10(name = "分離距離 r [km]") +
    scaleYContinuous(name = "σ_JK / σ_Poisson") +
    ggtitle("ジャックナイフ / Poisson 誤差比",
            ">> 1 → 大スケール構造が誤差を支配") +
    ggsize(800, 300)

pXi


<path d="M0.0 152.92891960830693 L0.0 152.92891960830693 L16.239938563366536 134.02937911378024 L32.47987712673313 161.88680956873907 L48.71981569009961 181.18982968608157 L64.9597542534662 182.41266673337583 L81.19969281683274 196.90785979652014 L97.43963138019927 205.95384961223584 L113.67956994356581 220.8784927126116 L129.91950850693235 232.49513880527928 L146.15944707029888 243.7318535628496 L162.39938563366547 256.8133542348614 L178.639324197032 263.7734214173724 L194.87926276039855 269.13082793944085 L211.11920132376508 274.2504547663159 L227.35913988713162 278.3042078636571 L243.59907845049815 281.5323645323465 L259.8390170138647 283.45757964521295 L276.0789555772312 284.8162271313978 L292.31889414059776 284.8382342025218 L308.5588327039643 285.7492007194841 L324.79877126733084 286.2025679464849 L341.0387098306975 286.8421114487802 L357.2786483940639 287.1045304968746 L373.51858695743056 286.72846772118595 L389.758525520797 287.3473320156783 L405.99846408416363 287.50046178476407 L422.23840264753017 287.42603339036543 L438.4783412108966 287.39684666692796 L454.7182797742631 287.51086722524246 L470.9582183376299 287.3362353539906 L487.1981569009963 285.96715794692466 L503.43809546436273 286.92590470760007 L519.6780340277294 287.6098036911168 L535.917972591096 287.7587097359237 L552.1579111544625 287.6336378060815 L568.397849717829 287.4714886680893 L584.6377882811955 287.84406488427305 L600.8777268445622 288.0301613787913 L600.8777268445622 288.03049381264725 L584.6377882811955 287.8446465579805 L568.397849717829 287.47260336708507 L552.1579111544625 287.634594473961 L535.917972591096 287.75955457277684 L519.6780340277294 287.6109964430577 L503.43809546436273 286.9285650934554 L487.1981569009963 285.9722469998574 L470.9582183376299 287.33859274463384 L454.7182797742631 287.5130473758608 L438.4783412108966 287.399685549951 L422.23840264753017 287.4291607081603 L405.99846408416363 287.5037211524136 L389.758525520797 287.3517804423361 L373.51858695743056 286.73689470050175 L357.2786483940639 287.1118579208045 L341.0387098306975 286.85237465282626 L324.79877126733084 286.219555815927 L308.5588327039643 285.77296022856706 L292.31889414059776 284.87546382282966 L276.0789555772312 284.8592531835621 L259.8390170138647 283.52653772253024 L243.59907845049815 281.6440461942502 L227.35913988713162 278.4939027510529 L211.11920132376508 274.5584550848197 L194.87926276039855 269.61440597983847 L178.639324197032 264.4886629766055 L162.39938563366547 257.8526740774298 L146.15944707029888 245.46672961629315 L129.91950850693235 234.95883154234298 L113.67956994356581 224.29994820267015 L97.43963138019927 210.85433162324568 L81.19969281683274 203.04610595703326 L64.9597542534662 190.6723516768821 L48.71981569009961 190.5029719899095 L32.47987712673313 174.80220984497947 L16.239938563366536 152.70039777606024 L0.0 170.60271877499383 Z" fill="rgb(70,130,180)" stroke-width="1.0" fill-opacity="0.14901960784313725">
 
 
 
 <path d="M0.0 63.8592672991108 L0.0 63.8592672991108 L16.239938563366536 13.727272727272691 L32.47987712673313 59.60247516679763 L48.71981569009961 93.11156525331637 L64.9597542534662 94.85393789981927 L81.19969281683274 115.61216826689721 L97.43963138019927 134.34618744249525 L113.67956994356581 163.6325411528155 L129.91950850693235 183.25560794046606 L146.15944707029888 204.56976925452398 L162.39938563366547 227.85215603663684 L178.639324197032 242.4843384994979 L194.87926276039855 253.73830853231487 L211.11920132376508 263.7861249985078 L227.35913988713162 272.7986559815087 L243.59907845049815 278.7582642512398 L259.8390170138647 281.5051216050235 L276.0789555772312 283.00095962983465 L292.31889414059776 282.84641530477546 L308.5588327039643 284.7215366073872 L324.79877126733084 285.39111876181215 L341.0387098306975 286.24855999146257 L357.2786483940639 286.6641234946328 L373.51858695743056 286.2223815949 L389.758525520797 287.12093702882424 L405.99846408416363 287.09492460401054 L422.23840264753017 287.1230719695454 L438.4

In [38]:
pRatio

<path d="M0.0 191.54810602143215 L0.0 191.54810602143215 L19.31357104306187 191.19437254554464 L38.6271420861238 190.82231732726626 L57.94071312918561 190.43475233133793 L77.25428417224754 190.14661669897342 L96.56785521530946 189.48043779099154 L115.88142625837133 189.13561619661084 L135.19499730143326 188.60156940696027 L154.50856834449507 187.78135333980447 L173.822139387557 187.1292969559113 L193.13571043061893 185.79559518551497 L212.44928147368086 185.31692540968385 L231.76285251674273 184.79636136235163 L251.07642355980465 184.25588950620968 L270.3899946028665 185.5038437059264 L289.7035656459284 186.5582430234553 L309.01713668899026 185.68264972324266 L328.33070773205213 182.18559514079413 L347.644278775114 179.3350419391485 L366.957849818176 181.91775443692765 L386.27142086123786 180.78024076976226 L405.5849919042997 178.24329433004291 L424.8985629473616 177.67097010798076 L444.2121339904236 177.68318609782955 L463.52570503348545 179.99221329317427 L482.8392760765473 161.46171469616718 L502.1528471196092 168.4038349018264 L521.4664181626711 175.44140708451636 L540.7799892057329 157.0433216798712 L560.093560248795 157.72728273807758 L579.4071312918568 157.29105701669795 L598.7207023349185 127.510003667901 L618.0342733779806 127.21653111400872 L637.3478444210425 115.44516759065903 L656.6614154641044 112.53513893334237 L675.9749865071662 69.16806721265947 L695.2885575502281 105.35477034413489 L714.6021285932901 9.181818181818187 " fill="none" stroke-width="2.64" stroke="rgb(139,26,26)" stroke-opacity="1.0">
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 1,000 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 500 
 
 
 
 
 
 
 1,000 
 
 
 
 
 
 
 1,500 
 
 
 
 
 
 
 
 
 ジャックナイフ / Poisson 誤差比 
 
 
 
 
 >> 1 → 大スケール構造が誤差を支配 
 
 
 
 
 σ_JK / σ_Poisson 
 
 
 
 
 分離距離 r [km]

In [39]:
// --- ジャックナイフ誤差解析結果を CSV に出力 ---
// 出力: output/xi_jackknife.csv
run {
    val file = java.io.File("./output/xi_jackknife.csv")
    file.bufferedWriter().use { w ->
        w.appendLine("# Spatial jackknife (K=$K longitude patches)")
        w.appendLine("# sigma_jk = sqrt((K-1)/K * sum_k (xi_k - xi_mean)^2)")
        w.appendLine("r_km,xi_cc,sigma_poisson,sigma_jk,jk_over_poisson")
        (0 until nBins).forEach { b ->
            val xi = xiMasked[b].xi
            val sP = xiMasked[b].xiErr
            val sJ = sigmaJK[b]
            val ratio = if (!sP.isNaN() && !sJ.isNaN() && sP > 0) sJ / sP else Double.NaN
            w.appendLine("${rCenters[b]},$xi,$sP,$sJ,$ratio")
        }
    }
    println("保存: ${file.path}  ($nBins 行)")
}

// Retail BAO スケールでの要約
val baoIdx = (0 until nBins).minByOrNull { abs(rCenters[it] - 666.0) }!!
println()
println("=== Retail BAO (r ≈ ${rCenters[baoIdx].toInt()} km) での誤差評価 ===")
println("  ξ_CC         = %.4f".format(xiMasked[baoIdx].xi))
println("  σ_Poisson    = %.4f  → S/N_Poisson = %.1f".format(
    xiMasked[baoIdx].xiErr,
    if (xiMasked[baoIdx].xiErr > 0) xiMasked[baoIdx].xi / xiMasked[baoIdx].xiErr else Double.NaN))
println("  σ_JK         = %.4f  → S/N_JK      = %.1f".format(
    sigmaJK[baoIdx],
    if (sigmaJK[baoIdx] > 0 && !sigmaJK[baoIdx].isNaN()) xiMasked[baoIdx].xi / sigmaJK[baoIdx] else Double.NaN))
println("  σ_JK / σ_P  = %.2fx".format(
    if (xiMasked[baoIdx].xiErr > 0 && !sigmaJK[baoIdx].isNaN()) sigmaJK[baoIdx] / xiMasked[baoIdx].xiErr else Double.NaN))

保存: ./output/xi_jackknife.csv  (38 行)

=== Retail BAO (r ≈ 666 km) での誤差評価 ===
  ξ_CC         = 3.2306
  σ_Poisson    = 0.0046  → S/N_Poisson = 696.6
  σ_JK         = 1.3121  → S/N_JK      = 2.5
  σ_JK / σ_P  = 282.94x


## 都市別店舗数比較：Coles / Woolworths 非対称 bias の検証

3エージェント討論で提起された仮説を数値検証する：

| 仮説 | 内容 | 検証方法 |
|---|---|---|
| A（BAO 非対称） | Melbourne: Coles 優位、Sydney: Woolworths 優位 | 都市圏内の C/W 比 |
| B（114 km バンプ）| Newcastle: Coles 特異集中 | Newcastle 圏の C/W 比 |
| C（239 km バンプ）| Canberra: Woolworths 優位 | Canberra 圏の C/W 比 |

都市境界は矩形バウンディングボックスで近似（lat/lon 範囲）。

In [40]:
// --- 都市別バウンディングボックス定義 ---
// ⚠️ 前提: Cell 2 (dataPoints, nD) と Cell 19 (woolworthsPoints, nW) が実行済みであること

data class CityBox(
    val name: String,
    val latMin: Double, val latMax: Double,
    val lonMin: Double, val lonMax: Double
)

val cities = listOf(
    CityBox("Sydney",     -34.2, -33.4, 150.5, 151.4),
    CityBox("Melbourne",  -38.2, -37.4, 144.5, 145.8),
    CityBox("Brisbane",   -27.8, -27.2, 152.7, 153.3),
    CityBox("Adelaide",   -35.2, -34.6, 138.4, 138.9),
    CityBox("Perth",      -32.2, -31.7, 115.7, 116.1),
    CityBox("Canberra",   -35.5, -35.2, 148.9, 149.3),
    CityBox("Newcastle",  -33.1, -32.7, 151.5, 151.9),
    CityBox("Gold Coast", -28.2, -27.9, 153.3, 153.6),
    CityBox("Hobart",     -43.0, -42.7, 147.0, 147.5),
    CityBox("Darwin",     -12.6, -12.3, 130.8, 131.1)
)

fun countInBox(points: List<Point>, box: CityBox): Int =
    points.count { p ->
        p.lat in box.latMin..box.latMax && p.lon in box.lonMin..box.lonMax
    }

data class CityResult(
    val name: String,
    val nColes: Int,
    val nWoolworths: Int,
    val ratio: Double   // Coles / Woolworths (> 1 → Coles 優勢)
)

val cityResults: List<CityResult> = cities.map { box ->
    val nC = countInBox(dataPoints, box)
    val nW = countInBox(woolworthsAsPoints, box)
    CityResult(box.name, nC, nW, if (nW > 0) nC.toDouble() / nW else Double.NaN)
}

println("%-12s  %6s  %6s  %10s  %s".format("都市", "Coles", "Woolworths", "C/W 比", "優位チェーン"))
println("-".repeat(55))
cityResults.forEach { r ->
    val winner = when {
        r.ratio.isNaN() -> "—"
        r.ratio > 1.05  -> "Coles"
        r.ratio < 0.95  -> "Woolworths"
        else            -> "拮抗"
    }
    println("%-12s  %6d  %6d  %10.3f  %s".format(
        r.name, r.nColes, r.nWoolworths, r.ratio, winner))
}

println()
println("全国合計: Coles $nD 店舗 / Woolworths $nW 店舗  (比率 = ${"%.3f".format(nD.toDouble() / nW)})")

都市             Coles  Woolworths       C/W 比  優位チェーン
-------------------------------------------------------
Sydney           114     160       0.713  Woolworths
Melbourne        137     177       0.774  Woolworths
Brisbane          56     106       0.528  Woolworths
Adelaide          34      47       0.723  Woolworths
Perth             53      65       0.815  Woolworths
Canberra          11      16       0.688  Woolworths
Newcastle         16      21       0.762  Woolworths
Gold Coast        20      22       0.909  Woolworths
Hobart             5      15       0.333  Woolworths
Darwin             6       8       0.750  Woolworths

全国合計: Coles 685 店舗 / Woolworths 1039 店舗  (比率 = 0.659)


In [41]:
// --- 都市別 Coles/Woolworths 比率バープロット ---
val validCities = cityResults.filter { !it.ratio.isNaN() && (it.nColes + it.nWoolworths) >= 3 }

val cityNames = validCities.map { it.name }
val ratios    = validCities.map { it.ratio }
val nColes    = validCities.map { it.nColes.toDouble() }
val nWools    = validCities.map { it.nWoolworths.toDouble() }

// 全国比率 (baseline)
val nationalRatio = nD.toDouble() / nW

// bar plot: C/W 比
letsPlot(mapOf("city" to cityNames, "ratio" to ratios)) +
    geomBar(stat = Stat.identity, width = 0.6) {
        x = "city"; y = "ratio"; fill = "ratio"
    } +
    geomHLine(yintercept = nationalRatio, linetype = "dashed", color = "#CC4444", size = 1.0) +
    geomHLine(yintercept = 1.0, linetype = "dotted", color = "#666666") +
    scaleXDiscrete(name = "都市") +
    scaleYContinuous(name = "Coles / Woolworths 店舗数比") +
    scaleFillGradient2(low = "#4682B4", mid = "#EEEEEE", high = "#E87040", midpoint = 1.0) +
    ggtitle("都市別 Coles/Woolworths 比率",
            "赤破線: 全国比 (${String.format("%.3f", nationalRatio)}) | 点線: 等数 (1.0) | > 1 → Coles 優勢") +
    ggsize(800, 400)

Sydney 
 
 
 
 
 
 
 
 
 Melbourne 
 
 
 
 
 
 
 
 
 Brisbane 
 
 
 
 
 
 
 
 
 Adelaide 
 
 
 
 
 
 
 
 
 Perth 
 
 
 
 
 
 
 
 
 Canberra 
 
 
 
 
 
 
 
 
 Newcastle 
 
 
 
 
 
 
 
 
 Gold Coast 
 
 
 
 
 
 
 
 
 Hobart 
 
 
 
 
 
 
 
 
 Darwin 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 0.8 
 
 
 
 
 
 
 1 
 
 
 
 
 
 
 
 
 都市別 Coles/Woolworths 比率 
 
 
 
 
 赤破線: 全国比 (0.659) | 点線: 等数 (1.0) | > 1 → Coles 優勢 
 
 
 
 
 Coles / Woolworths 店舗数比 
 
 
 
 
 都市 
 
 
 
 
 
 
 
 
 ratio 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 
 
 0.5 
 
 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 
 
 0.7 
 
 
 
 
 
 
 
 
 0.8 
 
 
 
 
 
 
 
 
 0.9

In [42]:
// --- 仮説検証サマリー ---
println("=== 仮説検証結果 ===")
println()

val cities_map = cityResults.associateBy { it.name }

fun testHypothesis(label: String, cityName: String, expectedWinner: String) {
    val r = cities_map[cityName] ?: return
    val actual = when {
        r.ratio > 1.05 -> "Coles"
        r.ratio < 0.95 -> "Woolworths"
        else -> "拮抗"
    }
    val match = if (actual == expectedWinner) "✓ 支持" else "✗ 否定"
    println("[$match]  $label")
    println("         $cityName: Coles=${r.nColes}, Woolworths=${r.nWoolworths}, C/W=${String.format("%.3f", r.ratio)} → $actual 優勢")
    println()
}

testHypothesis("仮説 A-1: BAO（666 km）— Melbourne で Coles 優勢", "Melbourne", "Coles")
testHypothesis("仮説 A-2: BAO（666 km）— Sydney で Woolworths 優勢",  "Sydney", "Woolworths")
testHypothesis("仮説 B:   114 km バンプ — Newcastle で Coles 特異集中", "Newcastle", "Coles")
testHypothesis("仮説 C:   239 km バンプ — Canberra で Woolworths 優勢", "Canberra", "Woolworths")

println("=== 解釈 ===")
println("Coles/Woolworths の C/W 比が全国比 (${"%.3f".format(nD.toDouble() / nW)}) を上回る都市 → Coles の相対的集中")
println("C/W 比が全国比を下回る都市 → Woolworths の相対的集中")

=== 仮説検証結果 ===

[✗ 否定]  仮説 A-1: BAO（666 km）— Melbourne で Coles 優勢
         Melbourne: Coles=137, Woolworths=177, C/W=0.774 → Woolworths 優勢

[✓ 支持]  仮説 A-2: BAO（666 km）— Sydney で Woolworths 優勢
         Sydney: Coles=114, Woolworths=160, C/W=0.713 → Woolworths 優勢

[✗ 否定]  仮説 B:   114 km バンプ — Newcastle で Coles 特異集中
         Newcastle: Coles=16, Woolworths=21, C/W=0.762 → Woolworths 優勢

[✓ 支持]  仮説 C:   239 km バンプ — Canberra で Woolworths 優勢
         Canberra: Coles=11, Woolworths=16, C/W=0.688 → Woolworths 優勢

=== 解釈 ===
Coles/Woolworths の C/W 比が全国比 (0.659) を上回る都市 → Coles の相対的集中
C/W 比が全国比を下回る都市 → Woolworths の相対的集中
